In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)


def resolve_model_dir():
    from pathlib import Path

    candidates = [
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
        "/kaggle/input/qwen3-4b-grids15-sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3-4b-grids15-sft139/Transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
    ]
    for candidate in candidates:
        path = Path(candidate)
        if (path / "config.json").exists():
            print(f"Using model dir: {path}")
            return str(path)

    for config in Path("/kaggle/input").glob("**/config.json"):
        s = str(config).lower()
        if "qwen" in s and "grid" in s:
            print(f"Using discovered model dir: {config.parent}")
            return str(config.parent)

    roots = []
    try:
        roots = [p.name for p in Path("/kaggle/input").iterdir()]
    except Exception:
        pass
    raise FileNotFoundError(f"Could not locate Qwen grid model. input_roots={roots[:40]}")


def stable_seed_for_key(key, offset=0):
    base = sum((i + 1) * ord(ch) for i, ch in enumerate(str(key)))
    return (base + int(offset)) % (1024 ** 2)


ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "ÄŠ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # ðŸ”§ KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=resolve_model_dir(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=stable_seed_for_key(bk, os.getenv('ARC_AUG_SEED_OFFSET', 0)))
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode and os.getenv("ARC_FULL_PUBLIC_EVAL", "1") != "1":
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

## ArcGen verified operation-bank branch (experimental, symbolic, offline)

A third verified source next to Layer-ops v2 and Transformer-search v3: a 108-operation grid DSL (geometry, colour, objects,
lines, relational rules such as marker-recolouring, template stamping, symmetry/tiling repair) searched by a target-guided
beam search. It runs **in a background process while Qwen is working**, so it is not limited by the 240 s post-processing window.

A prediction is accepted only if (1) a program reproduces every demonstration, (2) leave-one-demo-out search with the same
operation skeleton predicts each held-out demo exactly, (3) programs found with different seeds agree on every query, and
(4) every query output is a valid grid that differs from its input. It may only fill attempt_2 when v2 and v3 accepted nothing;
Qwen's attempt_1 is never changed. Search is bounded, not exhaustive. Disable with `ARC_ARCGEN=0`.
Optional: attach `arcgen_gpt.pt` (the small program-proposal model) to bias the search; it is not required.

In [ ]:
import base64, io, tarfile, zlib
ARCGEN_B64 = "..."
tarfile.open(fileobj=io.BytesIO(zlib.decompress(base64.b64decode(ARCGEN_B64)))).extractall(".")
import sys; sys.path.insert(0, ".")
import arcgen; print("arcgen unpacked:", len(arcgen.OPS), "operations")

In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path

ARCGEN_ENABLED = os.getenv("ARC_ARCGEN", "1") == "1"
ARCGEN_OUT = "/kaggle/working/arcgen_predictions.jsonl"
ARCGEN_PROC = None
if ARCGEN_ENABLED:
    _root = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")
    _file = _root / ("arc-agi_test_challenges.json" if os.getenv("KAGGLE_IS_COMPETITION_RERUN")
                     else "arc-agi_evaluation_challenges.json")
    _model = next(iter(sorted(Path("/kaggle/input").rglob("arcgen_gpt.pt"))), None) if Path("/kaggle/input").exists() else None
    if _file.is_file():
        _cmd = [sys.executable, "-m", "arcgen.kaggle_branch", "--challenges", str(_file), "--out", ARCGEN_OUT,
                "--workers", os.getenv("ARC_ARCGEN_WORKERS", "2"), "--seconds", os.getenv("ARC_ARCGEN_SECONDS", "45"),
                "--deadline", str(global_end_time - 300)] + (["--model", str(_model)] if _model else [])
        ARCGEN_PROC = subprocess.Popen(_cmd, stdout=open("arcgen_branch.log", "w"), stderr=subprocess.STDOUT,
                                       env={**os.environ, "PYTHONPATH": os.getcwd(), "OMP_NUM_THREADS": "1"})
        print("arcgen branch started in background:", " ".join(_cmd[2:6]), "| model:", _model)
    else:
        print("arcgen branch disabled: challenge file not found", _file)

In [ ]:
!PYTHONHASHSEED=260618 ARC_AUG_SEED_OFFSET=260618 UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

## Transformer-first search v3 (experimental)

Attach **layer_relation_transformer_search_v3.pt** as a Kaggle Dataset, alongside the original Qwen assets. 34 executable operation families; bounded programs of up to three steps. The trained transformer ranks the next operation family BEFORE parameter enumeration at each visited state. Parameters remain symbolic. Exact demo fit, independent leave-one-demo-out search and query consensus are required. Search is bounded, not exhaustive. No external API/internet/training is used by this new branch. Old Qwen attempt_1 and accepted v2 predictions are preserved.

Public evaluation now attempts ALL tasks by default. Set environment variable ARC_FULL_PUBLIC_EVAL=0 before running starter.py only for a four-task smoke test; check run_coverage.json. Hidden rerun routing is unchanged. Synthetic metrics are not Kaggle accuracy; no score improvement is guaranteed.


In [ ]:
from pathlib import Path
Path('operation_mining').mkdir(exist_ok=True)


In [ ]:
%%writefile layer_relation_transformer.py
"""Graph-aware transformer for ordinal-blind ARC colour-layer reasoning.

This module is original research code.  It represents a grid as ten categorical
boolean layer nodes plus directed pairwise spatial relations.  The model
proposes symbolic programs; deterministic execution/verification remains a
separate stage.
"""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from itertools import product
import time
from typing import Any

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset


CANONICAL_SHAPE_SIZE = 8
MAX_OBJECTS = 16
BASE_NODE_FEATURE_NAMES = (
    "present", "area_fraction", "component_fraction", "bbox_height", "bbox_width",
    "centroid_row", "centroid_col", "bbox_density", "touches_border", "hole_fraction",
    "horizontal_symmetry", "vertical_symmetry", "rot180_symmetry", "row_span",
    "column_span", "is_singleton",
)
ORIENTED_SHAPE_FEATURE_NAMES = tuple(
    f"oriented_shape_{r}_{c}" for r in range(CANONICAL_SHAPE_SIZE) for c in range(CANONICAL_SHAPE_SIZE))
CANONICAL_SHAPE_FEATURE_NAMES = tuple(
    f"canonical_shape_{r}_{c}" for r in range(CANONICAL_SHAPE_SIZE) for c in range(CANONICAL_SHAPE_SIZE))
NODE_FEATURE_NAMES = BASE_NODE_FEATURE_NAMES + ORIENTED_SHAPE_FEATURE_NAMES + CANONICAL_SHAPE_FEATURE_NAMES

EDGE_FEATURE_NAMES = (
    "both_present", "bbox_iou", "touching_4", "inverse_manhattan_distance",
    "source_inside_target_bbox", "target_inside_source_bbox", "same_shape_d4",
    "same_area", "row_overlap", "column_overlap", "same_centroid_row",
    "same_centroid_column", "source_above", "source_below", "source_left",
    "source_right", "area_ratio", "component_ratio",
)

OBJECT_BASE_FEATURE_NAMES = (
    "present", "area_fraction", "bbox_height", "bbox_width", "centroid_row", "centroid_col",
    "bbox_density", "touches_border", "hole_fraction", "horizontal_symmetry", "vertical_symmetry",
    "rot180_symmetry", "row_span", "column_span", "is_singleton", "layer_area_share",
    "area_rank_within_layer", "component_count_fraction",
)
OBJECT_ORIENTED_SHAPE_FEATURE_NAMES = tuple(
    f"oriented_shape_{r}_{c}" for r in range(CANONICAL_SHAPE_SIZE) for c in range(CANONICAL_SHAPE_SIZE))
OBJECT_CANONICAL_SHAPE_FEATURE_NAMES = tuple(
    f"canonical_shape_{r}_{c}" for r in range(CANONICAL_SHAPE_SIZE) for c in range(CANONICAL_SHAPE_SIZE))
OBJECT_FEATURE_NAMES = (OBJECT_BASE_FEATURE_NAMES + OBJECT_ORIENTED_SHAPE_FEATURE_NAMES
                        + OBJECT_CANONICAL_SHAPE_FEATURE_NAMES)
OBJECT_EDGE_FEATURE_NAMES = (
    "both_present", "same_colour_layer", "bbox_iou", "touching_4", "touching_8",
    "inverse_manhattan_distance", "source_inside_target_bbox", "target_inside_source_bbox",
    "same_shape_exact", "same_shape_d4", "same_area", "row_overlap", "column_overlap",
    "same_centroid_row", "same_centroid_column", "source_above", "source_below",
    "source_left", "source_right", "area_ratio",
)

PROGRAM_OPS = (
    "identity",
    "move_toward", "move_away", "move_until_touch", "move_away_until_border",
    "align_row", "align_column",
    "recolor_as_context", "swap_source_context_colors",
    "fill_bbox", "outline_bbox", "fill_holes", "delete_source",
    "rotate90", "rotate180", "reflect_left_right", "reflect_up_down",
    "copy_toward", "connect_source_to_context",
)
TARGET_REQUIRED_OPS = frozenset({"move_toward", "move_away", "move_until_touch", "move_away_until_border",
                                 "align_row", "align_column", "recolor_as_context", "swap_source_context_colors",
                                 "copy_toward", "connect_source_to_context"})
TRANSFORM_REQUIRED_OPS = frozenset({"rotate90", "rotate180", "reflect_left_right", "reflect_up_down"})
DELTA_REQUIRED_OPS = frozenset({"move_toward", "move_away", "move_until_touch", "move_away_until_border",
                                "align_row", "align_column", "copy_toward"})
OBJECT_SELECTORS = (
    "all", "largest", "smallest", "topmost", "bottommost", "leftmost", "rightmost",
    "border", "interior", "has_hole", "unique_area", "repeated_area", "unique_shape", "repeated_shape",
)
OBJECT_STEP_OPS = (
    "keep_selected", "delete_selected", "recolor_selected", "crop_selected",
    "fill_selected_bbox", "outline_selected_bbox", "transform_selected", "gravity_selected",
)
LAYER_ROLES = (
    "only_foreground", "rarest", "most_frequent", "fewest_components", "most_components",
    "smallest_object_layer", "largest_object_layer", "has_hole", "border_touching", "interior_only",
)


def to_layers(grid: Any) -> np.ndarray:
    grid = np.asarray(grid, dtype=np.int8)
    if grid.ndim != 2 or np.any(grid < 0) or np.any(grid > 9):
        raise ValueError("ARC grid must be H×W with categorical values 0..9")
    return np.arange(10, dtype=np.int8)[:, None, None] == grid[None]


def _components(mask: np.ndarray) -> list[list[tuple[int, int]]]:
    mask = np.asarray(mask, bool); h, w = mask.shape; seen = np.zeros_like(mask)
    result = []
    for sr, sc in np.argwhere(mask):
        sr, sc = int(sr), int(sc)
        if seen[sr, sc]: continue
        stack = [(sr, sc)]; seen[sr, sc] = True; cells = []
        while stack:
            r, c = stack.pop(); cells.append((r, c))
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < h and 0 <= nc < w and mask[nr, nc] and not seen[nr, nc]:
                    seen[nr, nc] = True; stack.append((nr, nc))
        result.append(cells)
    return result


def _bbox(mask: np.ndarray) -> tuple[int, int, int, int] | None:
    pts = np.argwhere(mask)
    if not len(pts): return None
    r0, c0 = pts.min(0); r1, c1 = pts.max(0)
    return int(r0), int(r1), int(c0), int(c1)


def _crop(mask: np.ndarray) -> np.ndarray:
    box = _bbox(mask)
    if box is None: return np.zeros((1, 1), bool)
    r0, r1, c0, c1 = box
    return np.asarray(mask, bool)[r0:r1 + 1, c0:c1 + 1]


def _holes(mask: np.ndarray) -> int:
    mask = np.asarray(mask, bool); h, w = mask.shape; seen = np.zeros_like(mask)
    holes = 0
    for sr in range(h):
        for sc in range(w):
            if mask[sr, sc] or seen[sr, sc]: continue
            stack = [(sr, sc)]; seen[sr, sc] = True; boundary = False
            while stack:
                r, c = stack.pop(); boundary |= r in (0, h - 1) or c in (0, w - 1)
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < h and 0 <= nc < w and not mask[nr, nc] and not seen[nr, nc]:
                        seen[nr, nc] = True; stack.append((nr, nc))
            holes += int(not boundary)
    return holes


def _d4(mask: np.ndarray, code: int) -> np.ndarray:
    if code < 4: return np.rot90(mask, code)
    return np.rot90(np.fliplr(mask), code - 4)


def _fit_shape_bitmap(mask: np.ndarray, size: int = CANONICAL_SHAPE_SIZE) -> np.ndarray:
    mask = _crop(mask); h, w = mask.shape
    if max(h, w) <= size:
        factor = max(1, size // max(h, w)); z = np.repeat(np.repeat(mask, factor, 0), factor, 1)
    else:
        rows = np.minimum((np.arange(size) * h / size).astype(int), h-1)
        cols = np.minimum((np.arange(size) * w / size).astype(int), w-1); z = mask[np.ix_(rows, cols)]
    canvas = np.zeros((size, size), bool); zh, zw = z.shape
    r0, c0 = (size-zh)//2, (size-zw)//2; canvas[r0:r0+zh, c0:c0+zw] = z
    return canvas


def _canonical_shape_bitmap(mask: np.ndarray) -> np.ndarray:
    variants = [_fit_shape_bitmap(_d4(_crop(mask), d)) for d in range(8)]
    return min(variants, key=lambda z: z.tobytes()).astype(np.float32)


def _oriented_shape_bitmap(mask: np.ndarray) -> np.ndarray:
    return _fit_shape_bitmap(_crop(mask)).astype(np.float32)


def _same_shape_d4(a: np.ndarray, b: np.ndarray) -> bool:
    a, b = _crop(a), _crop(b)
    return any((z := _d4(a, d)).shape == b.shape and np.array_equal(z, b) for d in range(8))


def _interval_overlap(a0: int, a1: int, b0: int, b1: int) -> float:
    inter = max(0, min(a1, b1) - max(a0, b0) + 1)
    union = max(a1, b1) - min(a0, b0) + 1
    return inter / max(1, union)


def _touching(a: np.ndarray, b: np.ndarray) -> bool:
    h, w = a.shape
    for r, c in np.argwhere(a):
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = int(r + dr), int(c + dc)
            if 0 <= nr < h and 0 <= nc < w and b[nr, nc]: return True
    return False


def _min_manhattan(a: np.ndarray, b: np.ndarray) -> int:
    ap, bp = np.argwhere(a), np.argwhere(b)
    if not len(ap) or not len(bp): return 999
    small, large = (ap, bp) if len(ap) <= len(bp) else (bp, ap)
    return min(int(np.abs(large - p).sum(axis=1).min()) for p in small)


def extract_layer_graph(grid: Any) -> tuple[np.ndarray, np.ndarray]:
    """Return node[10,N] and directed edge[10,10,R] features."""
    g = np.asarray(grid, dtype=np.int8); layers = to_layers(g); h, w = g.shape
    nodes = np.zeros((10, len(NODE_FEATURE_NAMES)), np.float32)
    desc = []
    for color, mask in enumerate(layers):
        pts = np.argwhere(mask); box = _bbox(mask); comps = _components(mask)
        if box is None:
            desc.append(None); continue
        r0, r1, c0, c1 = box; crop = _crop(mask); area = len(pts)
        base_features = [
            1.0, area / (h * w), min(len(comps), 10) / 10,
            (r1 - r0 + 1) / h, (c1 - c0 + 1) / w,
            float(pts[:, 0].mean() / max(1, h - 1)), float(pts[:, 1].mean() / max(1, w - 1)),
            area / crop.size, float(r0 == 0 or r1 == h - 1 or c0 == 0 or c1 == w - 1),
            min(_holes(crop), 5) / 5, float(np.array_equal(crop, np.flipud(crop))),
            float(np.array_equal(crop, np.fliplr(crop))), float(np.array_equal(crop, np.rot90(crop, 2))),
            (r1 - r0 + 1) / h, (c1 - c0 + 1) / w, float(area == 1),
        ]
        nodes[color] = np.asarray(base_features + _oriented_shape_bitmap(mask).ravel().tolist()
                                  + _canonical_shape_bitmap(mask).ravel().tolist(), np.float32)
        desc.append({"mask": mask, "box": box, "area": area, "components": len(comps),
                     "centroid": pts.mean(0)})

    edges = np.zeros((10, 10, len(EDGE_FEATURE_NAMES)), np.float32)
    for i, j in product(range(10), repeat=2):
        a, b = desc[i], desc[j]
        if a is None or b is None: continue
        ar0, ar1, ac0, ac1 = a["box"]; br0, br1, bc0, bc1 = b["box"]
        inter_h = max(0, min(ar1, br1) - max(ar0, br0) + 1)
        inter_w = max(0, min(ac1, bc1) - max(ac0, bc0) + 1)
        inter = inter_h * inter_w
        aa, ba = (ar1-ar0+1)*(ac1-ac0+1), (br1-br0+1)*(bc1-bc0+1)
        d = _min_manhattan(a["mask"], b["mask"])
        edges[i, j] = np.array([
            1, inter / max(1, aa + ba - inter), float(_touching(a["mask"], b["mask"])),
            1 / (1 + d), float(br0 <= ar0 <= ar1 <= br1 and bc0 <= ac0 <= ac1 <= bc1),
            float(ar0 <= br0 <= br1 <= ar1 and ac0 <= bc0 <= bc1 <= ac1),
            float(_same_shape_d4(a["mask"], b["mask"])), float(a["area"] == b["area"]),
            _interval_overlap(ar0, ar1, br0, br1), _interval_overlap(ac0, ac1, bc0, bc1),
            float(abs(a["centroid"][0]-b["centroid"][0]) < .5),
            float(abs(a["centroid"][1]-b["centroid"][1]) < .5),
            float(ar1 < br0), float(ar0 > br1), float(ac1 < bc0), float(ac0 > bc1),
            min(a["area"], b["area"]) / max(a["area"], b["area"]),
            min(a["components"], b["components"]) / max(1, a["components"], b["components"]),
        ], np.float32)
    return nodes, edges


def _object_background(grid: np.ndarray) -> int:
    border = np.concatenate((grid[0], grid[-1], grid[:, 0], grid[:, -1]))
    return int(np.argmax(np.bincount(border, minlength=10)))


def _touching_8(a: np.ndarray, b: np.ndarray) -> bool:
    h, w = a.shape
    b_cells = {tuple(map(int, p)) for p in np.argwhere(b)}
    return any((int(r + dr), int(c + dc)) in b_cells for r, c in np.argwhere(a)
               for dr in (-1, 0, 1) for dc in (-1, 0, 1) if dr or dc)


def extract_object_graph(grid: Any, max_objects: int = MAX_OBJECTS) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Return object nodes, directed edges, colour membership, and padding mask.

    Objects are monochrome 4-connected components. Token order is geometric,
    so relabelling colours does not alter object order or inject ordinality.
    """
    g = np.asarray(grid, dtype=np.int8)
    if g.ndim != 2 or np.any(g < 0) or np.any(g > 9):
        raise ValueError("ARC grid must be H×W with categorical values 0..9")
    h, w = g.shape; bg = _object_background(g); objects = []
    for colour in range(10):
        if colour == bg:
            continue
        layer_mask = g == colour
        components = _components(layer_mask)
        layer_area = max(1, int(layer_mask.sum())); count = len(components)
        ranked = {tuple(sorted(cells)): rank for rank, cells in enumerate(
            sorted(components, key=lambda z: (-len(z), min(z))))}
        for cells in components:
            mask = np.zeros_like(layer_mask)
            rr, cc = zip(*cells); mask[np.asarray(rr), np.asarray(cc)] = True
            r0, r1, c0, c1 = _bbox(mask); crop = _crop(mask); area = len(cells)
            rank = ranked[tuple(sorted(cells))]
            base = [
                1.0, area/(h*w), (r1-r0+1)/h, (c1-c0+1)/w,
                float(np.mean(rr)/max(1, h-1)), float(np.mean(cc)/max(1, w-1)),
                area/crop.size, float(r0 == 0 or r1 == h-1 or c0 == 0 or c1 == w-1),
                min(_holes(crop), 5)/5, float(np.array_equal(crop, np.flipud(crop))),
                float(np.array_equal(crop, np.fliplr(crop))), float(np.array_equal(crop, np.rot90(crop, 2))),
                (r1-r0+1)/h, (c1-c0+1)/w, float(area == 1), area/layer_area,
                rank/max(1, count-1) if count > 1 else 0.0, min(count, 16)/16,
            ]
            feature = np.asarray(base + _oriented_shape_bitmap(mask).ravel().tolist()
                                 + _canonical_shape_bitmap(mask).ravel().tolist(), np.float32)
            objects.append({"colour": colour, "mask": mask, "box": (r0, r1, c0, c1),
                            "area": area, "centroid": np.asarray((np.mean(rr), np.mean(cc))),
                            "feature": feature})
    if len(objects) > max_objects:
        objects = sorted(objects, key=lambda z: (-z["area"], z["box"]))[:max_objects]
    objects.sort(key=lambda z: (z["box"][0], z["box"][2], z["box"][1], z["box"][3], -z["area"]))
    node = np.zeros((max_objects, len(OBJECT_FEATURE_NAMES)), np.float32)
    edge = np.zeros((max_objects, max_objects, len(OBJECT_EDGE_FEATURE_NAMES)), np.float32)
    membership = np.zeros((max_objects, 10), np.float32)
    valid = np.zeros(max_objects, bool)
    for i, obj in enumerate(objects):
        node[i] = obj["feature"]; membership[i, obj["colour"]] = 1; valid[i] = True
    for i, a in enumerate(objects):
        ar0, ar1, ac0, ac1 = a["box"]
        for j, b in enumerate(objects):
            br0, br1, bc0, bc1 = b["box"]
            inter_h = max(0, min(ar1, br1)-max(ar0, br0)+1)
            inter_w = max(0, min(ac1, bc1)-max(ac0, bc0)+1); inter = inter_h*inter_w
            aa, ba = (ar1-ar0+1)*(ac1-ac0+1), (br1-br0+1)*(bc1-bc0+1)
            acrop, bcrop = _crop(a["mask"]), _crop(b["mask"])
            same_exact = acrop.shape == bcrop.shape and np.array_equal(acrop, bcrop)
            edge[i, j] = np.asarray([
                1, float(a["colour"] == b["colour"]), inter/max(1, aa+ba-inter),
                float(_touching(a["mask"], b["mask"])), float(_touching_8(a["mask"], b["mask"])),
                1/(1+_min_manhattan(a["mask"], b["mask"])),
                float(br0 <= ar0 <= ar1 <= br1 and bc0 <= ac0 <= ac1 <= bc1),
                float(ar0 <= br0 <= br1 <= ar1 and ac0 <= bc0 <= bc1 <= ac1),
                float(same_exact), float(_same_shape_d4(a["mask"], b["mask"])), float(a["area"] == b["area"]),
                _interval_overlap(ar0, ar1, br0, br1), _interval_overlap(ac0, ac1, bc0, bc1),
                float(abs(a["centroid"][0]-b["centroid"][0]) < .5),
                float(abs(a["centroid"][1]-b["centroid"][1]) < .5),
                float(ar1 < br0), float(ar0 > br1), float(ac1 < bc0), float(ac0 > bc1),
                min(a["area"], b["area"])/max(a["area"], b["area"]),
            ], np.float32)
    return node, edge, membership, valid


class RelationAttentionBlock(nn.Module):
    def __init__(self, dim: int, edge_dim: int, heads: int = 4, dropout: float = .1):
        super().__init__(); assert dim % heads == 0
        self.heads, self.head_dim = heads, dim // heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, 3 * dim); self.edge_bias = nn.Sequential(nn.Linear(edge_dim, dim), nn.GELU(), nn.Linear(dim, heads))
        self.out = nn.Linear(dim, dim); self.drop = nn.Dropout(dropout)
        self.ff = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(4*dim, dim))

    def forward(self, x: torch.Tensor, edge: torch.Tensor, node_mask: torch.Tensor | None = None) -> torch.Tensor:
        z = self.norm1(x); b, n, d = z.shape
        q, k, v = self.qkv(z).view(b, n, 3, self.heads, self.head_dim).unbind(2)
        logits = torch.einsum("bihd,bjhd->bhij", q, k) / self.head_dim**.5
        logits = logits + self.edge_bias(edge).permute(0, 3, 1, 2)
        if node_mask is not None:
            logits = logits.masked_fill(~node_mask[:, None, None, :].bool(), -1e4)
        attn = logits.softmax(-1)
        mixed = torch.einsum("bhij,bjhd->bihd", attn, v).reshape(b, n, d)
        x = x + self.drop(self.out(mixed))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x if node_mask is None else x * node_mask[..., None]


class LayerGraphEncoder(nn.Module):
    def __init__(self, dim: int = 128, depth: int = 3, heads: int = 4, categorical_color_embeddings: bool = True):
        super().__init__()
        self.node_proj = nn.Linear(len(NODE_FEATURE_NAMES), dim)
        self.color_embedding = nn.Embedding(10, dim) if categorical_color_embeddings else None
        self.blocks = nn.ModuleList([RelationAttentionBlock(dim, len(EDGE_FEATURE_NAMES), heads) for _ in range(depth)])
        self.pool_query = nn.Parameter(torch.randn(dim) / dim**.5); self.norm = nn.LayerNorm(dim)

    def forward(self, node: torch.Tensor, edge: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.node_proj(node)
        if self.color_embedding is not None:
            ids = torch.arange(10, device=x.device)
            x = x + self.color_embedding(ids)[None]
        for block in self.blocks: x = block(x, edge)
        x = self.norm(x); present = node[..., 0] > .5
        scores = torch.einsum("bnd,d->bn", x, self.pool_query).masked_fill(~present, -1e4)
        pooled = torch.einsum("bn,bnd->bd", scores.softmax(-1), x)
        return x, pooled


class ObjectGraphEncoder(nn.Module):
    """Permutation-equivariant encoder over component/island tokens."""
    def __init__(self, dim: int = 128, depth: int = 2, heads: int = 4):
        super().__init__()
        self.node_proj = nn.Linear(len(OBJECT_FEATURE_NAMES), dim)
        self.blocks = nn.ModuleList([
            RelationAttentionBlock(dim, len(OBJECT_EDGE_FEATURE_NAMES), heads) for _ in range(depth)])
        self.pool_query = nn.Parameter(torch.randn(dim) / dim**.5); self.norm = nn.LayerNorm(dim)

    def forward(self, node: torch.Tensor, edge: torch.Tensor, valid: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        valid = valid.bool(); x = self.node_proj(node) * valid[..., None]
        for block in self.blocks:
            x = block(x, edge, valid)
        x = self.norm(x) * valid[..., None]
        scores = torch.einsum("bnd,d->bn", x, self.pool_query).masked_fill(~valid, -1e4)
        weights = scores.softmax(-1) * valid
        weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-6)
        pooled = torch.einsum("bn,bnd->bd", weights, x)
        return x, pooled


class LayerRelationTransformer(nn.Module):
    """Encode demonstrations + query and propose an executable relation program."""
    def __init__(self, dim: int = 128, depth: int = 3, heads: int = 4):
        super().__init__(); self.dim = dim
        # No absolute colour embedding: arbitrary colour permutation simply
        # permutes nodes and preserves the model's computation.
        self.graph = LayerGraphEncoder(dim, depth, heads, categorical_color_embeddings=False)
        self.object_graph = ObjectGraphEncoder(dim, max(1, depth-1), heads)
        self.object_layer_fuse = nn.Sequential(nn.Linear(2*dim, dim), nn.GELU(), nn.Linear(dim, dim))
        explicit_change_dim = 4*len(NODE_FEATURE_NAMES) + 2*len(EDGE_FEATURE_NAMES)
        self.demo_proj = nn.Sequential(nn.Linear(4*dim + explicit_change_dim, 2*dim), nn.GELU(), nn.Linear(2*dim, dim))
        self.context_proj = nn.Sequential(nn.Linear(2*dim, 2*dim), nn.GELU(), nn.Linear(2*dim, dim))
        program_context_dim = 3*dim + len(EDGE_FEATURE_NAMES) + 2*len(BASE_NODE_FEATURE_NAMES)
        self.op_head = nn.Sequential(nn.Linear(program_context_dim, 2*dim), nn.GELU(), nn.Linear(2*dim, len(PROGRAM_OPS)))
        self.transform_head = nn.Linear(program_context_dim, 8)
        self.dr_head = nn.Linear(program_context_dim, 9); self.dc_head = nn.Linear(program_context_dim, 9)
        self.source_head = nn.Sequential(nn.Linear(3*dim, dim), nn.GELU(), nn.Linear(dim, 1))
        self.target_head = nn.Sequential(nn.Linear(3*dim + len(EDGE_FEATURE_NAMES) + 1, dim), nn.GELU(), nn.Linear(dim, 1))
        role_match_dim = len(BASE_NODE_FEATURE_NAMES) + len(CANONICAL_SHAPE_FEATURE_NAMES)
        self.role_projection = nn.Linear(role_match_dim, dim)
        self.role_value_projection = nn.Sequential(
            nn.Linear(4*len(NODE_FEATURE_NAMES), 2*dim), nn.GELU(), nn.Linear(2*dim, dim))
        self.candidate_head = nn.Sequential(nn.Linear(4*dim, 2*dim), nn.GELU(), nn.Linear(2*dim, 1))
        self.prototype_scale = nn.Parameter(torch.tensor(1.0))
        self.role_match_scale = nn.Parameter(torch.tensor(1.0))

    def _encode_many(self, nodes: torch.Tensor, edges: torch.Tensor,
                     object_nodes: torch.Tensor | None = None, object_edges: torch.Tensor | None = None,
                     object_membership: torch.Tensor | None = None, object_valid: torch.Tensor | None = None):
        shape = nodes.shape[:-2]; n = int(np.prod(shape))
        layer, pool = self.graph(nodes.reshape(n, 10, -1), edges.reshape(n, 10, 10, -1))
        if object_nodes is not None:
            objects, object_pool = self.object_graph(
                object_nodes.reshape(n, MAX_OBJECTS, -1),
                object_edges.reshape(n, MAX_OBJECTS, MAX_OBJECTS, -1),
                object_valid.reshape(n, MAX_OBJECTS))
            membership = object_membership.reshape(n, MAX_OBJECTS, 10)
            counts = membership.sum(1).clamp_min(1)[..., None]
            object_by_layer = torch.einsum("bnc,bnd->bcd", membership, objects) / counts
            has_objects = membership.sum(1) > 0
            fused = self.object_layer_fuse(torch.cat((layer, object_by_layer), -1))
            layer = layer + fused * has_objects[..., None]
            pool = pool + object_pool
        return layer.reshape(*shape, 10, self.dim), pool.reshape(*shape, self.dim)

    def _pair_embedding(self, in_layers, in_pool, out_layers, out_pool,
                        in_node, in_edge, out_node, out_edge):
        node_delta = out_node - in_node; edge_delta = out_edge - in_edge
        oriented_start = NODE_FEATURE_NAMES.index("oriented_shape_0_0")
        present_index = NODE_FEATURE_NAMES.index("present")
        area_index = NODE_FEATURE_NAMES.index("area_fraction")
        border_index = NODE_FEATURE_NAMES.index("touches_border")
        change_weight = node_delta[..., :oriented_start].abs().sum(-1)
        foreground = (1-in_node[..., area_index]).clamp_min(.05) * (1-.8*in_node[..., border_index])
        presence_loss = (-node_delta[..., present_index]).clamp_min(0)
        area_loss = (-node_delta[..., area_index]).clamp_min(0)
        change_weight = (change_weight + 4*presence_loss + 8*area_loss) * foreground
        change_weight = change_weight / change_weight.sum(-1, keepdim=True).clamp_min(1e-6)
        source_proto = (change_weight[..., None] * in_node).sum(-2)
        changed_input = source_proto
        changed_output = (change_weight[..., None] * out_node).sum(-2)
        explicit = torch.cat((node_delta.mean(-2), node_delta.abs().mean(-2), changed_input, changed_output,
                              edge_delta.mean((-3, -2)), edge_delta.abs().mean((-3, -2))), -1)
        relation = self.demo_proj(torch.cat((in_pool, out_pool, out_pool-in_pool,
                                  (out_layers-in_layers).abs().mean(-2), explicit), -1))
        return relation, source_proto, node_delta

    def forward(self, demo_in_node, demo_in_edge, demo_out_node, demo_out_edge,
                query_node, query_edge, demo_mask=None,
                demo_in_object_node=None, demo_in_object_edge=None, demo_in_object_membership=None, demo_in_object_valid=None,
                demo_out_object_node=None, demo_out_object_edge=None, demo_out_object_membership=None, demo_out_object_valid=None,
                query_object_node=None, query_object_edge=None, query_object_membership=None, query_object_valid=None,
                candidate_node=None, candidate_edge=None, candidate_object_node=None, candidate_object_edge=None,
                candidate_object_membership=None, candidate_object_valid=None):
        in_layers, ip = self._encode_many(demo_in_node, demo_in_edge, demo_in_object_node, demo_in_object_edge,
                                          demo_in_object_membership, demo_in_object_valid)
        out_layers, op = self._encode_many(demo_out_node, demo_out_edge, demo_out_object_node, demo_out_object_edge,
                                           demo_out_object_membership, demo_out_object_valid)
        qlayers, qp = self._encode_many(query_node, query_edge, query_object_node, query_object_edge,
                                        query_object_membership, query_object_valid)
        demo, source_proto_per_demo, node_delta = self._pair_embedding(
            in_layers, ip, out_layers, op, demo_in_node, demo_in_edge, demo_out_node, demo_out_edge)
        oriented_start = NODE_FEATURE_NAMES.index("oriented_shape_0_0")
        canonical_start = NODE_FEATURE_NAMES.index("canonical_shape_0_0")
        present_index = NODE_FEATURE_NAMES.index("present")
        if demo_mask is None:
            demo_context = demo.mean(1); source_proto = source_proto_per_demo.mean(1)
        else:
            weight = demo_mask.float() / demo_mask.float().sum(1, keepdim=True).clamp_min(1)
            demo_context = torch.einsum("bd,bdh->bh", weight, demo)
            source_proto = torch.einsum("bd,bdf->bf", weight, source_proto_per_demo)
        context = self.context_proj(torch.cat((demo_context, qp), -1))
        expanded = context[:, None].expand(-1, 10, -1)
        # Query-conditioned retrieval aligns same-role layers across examples
        # without assuming that their categorical colour ids are equal.
        demo_match_features = torch.cat((demo_in_node[..., :oriented_start], demo_in_node[..., canonical_start:]), -1)
        query_match_features = torch.cat((query_node[..., :oriented_start], query_node[..., canonical_start:]), -1)
        demo_keys = F.normalize(self.role_projection(demo_match_features), dim=-1)
        query_keys = F.normalize(self.role_projection(query_match_features), dim=-1)
        match_logits = torch.einsum("bqf,bdnf->bdqn", query_keys, demo_keys) * F.softplus(self.role_match_scale)
        demo_present = demo_in_node[..., present_index] > .5
        match_logits = match_logits.masked_fill(~demo_present[:, :, None, :], -1e4)
        match_weight = match_logits.softmax(-1)
        role_values = self.role_value_projection(torch.cat((demo_in_node, demo_out_node, node_delta, node_delta.abs()), -1))
        matched_per_demo = torch.einsum("bdqn,bdnh->bdqh", match_weight, role_values)
        matched_delta_per_demo = torch.einsum("bdqn,bdnf->bdqf", match_weight, node_delta)
        if demo_mask is None:
            matched_role = matched_per_demo.mean(1)
            matched_delta = matched_delta_per_demo.mean(1)
        else:
            matched_role = torch.einsum("bd,bdqh->bqh", weight, matched_per_demo)
            matched_delta = torch.einsum("bd,bdqf->bqf", weight, matched_delta_per_demo)
        pair = torch.cat((qlayers, expanded, matched_role), -1)
        source_match_features = torch.cat((source_proto[..., :oriented_start], source_proto[..., canonical_start:]), -1)
        proto = F.normalize(self.role_projection(source_match_features), dim=-1)
        prototype_score = torch.einsum("bnd,bd->bn", query_keys, proto) * F.softplus(self.prototype_scale)
        source_logits = self.source_head(pair).squeeze(-1) + prototype_score
        source_prob = source_logits.softmax(-1)
        source_to_target_relation = torch.einsum("bs,bstf->btf", source_prob, query_edge)
        target_pair = torch.cat((qlayers, expanded, matched_role, source_to_target_relation, source_prob[..., None]), -1)
        target_logits = self.target_head(target_pair).squeeze(-1)
        target_prob = target_logits.softmax(-1)
        source_change_summary = torch.einsum("bn,bnh->bh", source_prob, matched_role)
        target_change_summary = torch.einsum("bn,bnh->bh", target_prob, matched_role)
        source_raw_delta = torch.einsum("bn,bnf->bf", source_prob, matched_delta[..., :len(BASE_NODE_FEATURE_NAMES)])
        target_raw_delta = torch.einsum("bn,bnf->bf", target_prob, matched_delta[..., :len(BASE_NODE_FEATURE_NAMES)])
        role_relation = torch.einsum("bs,bt,bstf->bf", source_prob, target_prob, query_edge)
        program_context = torch.cat((context, source_change_summary, target_change_summary,
                                     role_relation, source_raw_delta, target_raw_delta), -1)
        result = {
            "op": self.op_head(program_context), "source": source_logits,
            "target": target_logits, "transform": self.transform_head(program_context),
            "dr": self.dr_head(program_context), "dc": self.dc_head(program_context),
        }
        if candidate_node is not None:
            candidate_layers, candidate_pool = self._encode_many(
                candidate_node, candidate_edge, candidate_object_node, candidate_object_edge,
                candidate_object_membership, candidate_object_valid)
            candidate_count = candidate_node.shape[1]
            qlayers_many = qlayers[:, None].expand(-1, candidate_count, -1, -1)
            qp_many = qp[:, None].expand(-1, candidate_count, -1)
            query_node_many = query_node[:, None].expand(-1, candidate_count, -1, -1)
            query_edge_many = query_edge[:, None].expand(-1, candidate_count, -1, -1, -1)
            candidate_relation, _, _ = self._pair_embedding(
                qlayers_many, qp_many, candidate_layers, candidate_pool,
                query_node_many, query_edge_many, candidate_node, candidate_edge)
            demo_many = demo_context[:, None].expand_as(candidate_relation)
            critic_features = torch.cat((demo_many, candidate_relation,
                                         (demo_many-candidate_relation).abs(), demo_many*candidate_relation), -1)
            result["candidate"] = self.candidate_head(critic_features).squeeze(-1)
        return result


def encode_task(task: dict, query_index: int = 0) -> dict[str, np.ndarray]:
    din_n, din_e, dout_n, dout_e = [], [], [], []
    din_on, din_oe, din_om, din_ov = [], [], [], []
    dout_on, dout_oe, dout_om, dout_ov = [], [], [], []
    for pair in task["train"]:
        n, e = extract_layer_graph(pair["input"]); on, oe, om, ov = extract_object_graph(pair["input"])
        din_n.append(n); din_e.append(e); din_on.append(on); din_oe.append(oe); din_om.append(om); din_ov.append(ov)
        n, e = extract_layer_graph(pair["output"]); on, oe, om, ov = extract_object_graph(pair["output"])
        dout_n.append(n); dout_e.append(e); dout_on.append(on); dout_oe.append(oe); dout_om.append(om); dout_ov.append(ov)
    qn, qe = extract_layer_graph(task["test"][query_index]["input"])
    qon, qoe, qom, qov = extract_object_graph(task["test"][query_index]["input"])
    return {"demo_in_node": np.stack(din_n), "demo_in_edge": np.stack(din_e),
            "demo_out_node": np.stack(dout_n), "demo_out_edge": np.stack(dout_e),
            "demo_in_object_node": np.stack(din_on), "demo_in_object_edge": np.stack(din_oe),
            "demo_in_object_membership": np.stack(din_om), "demo_in_object_valid": np.stack(din_ov),
            "demo_out_object_node": np.stack(dout_on), "demo_out_object_edge": np.stack(dout_oe),
            "demo_out_object_membership": np.stack(dout_om), "demo_out_object_valid": np.stack(dout_ov),
            "query_node": qn, "query_edge": qe, "query_object_node": qon, "query_object_edge": qoe,
            "query_object_membership": qom, "query_object_valid": qov}


def collate_relation_batch(samples: list[dict]) -> dict[str, torch.Tensor]:
    max_demo = max(s["demo_in_node"].shape[0] for s in samples); batch = {}
    for name in ("demo_in_node", "demo_in_edge", "demo_out_node", "demo_out_edge",
                 "demo_in_object_node", "demo_in_object_edge", "demo_in_object_membership", "demo_in_object_valid",
                 "demo_out_object_node", "demo_out_object_edge", "demo_out_object_membership", "demo_out_object_valid"):
        exemplar = samples[0][name]; shape = (len(samples), max_demo) + exemplar.shape[1:]
        value = np.zeros(shape, np.float32)
        for i, sample in enumerate(samples): value[i, :sample[name].shape[0]] = sample[name]
        batch[name] = torch.from_numpy(value)
    batch["demo_mask"] = torch.tensor([[j < s["demo_in_node"].shape[0] for j in range(max_demo)] for s in samples])
    for name in ("query_node", "query_edge", "query_object_node", "query_object_edge",
                 "query_object_membership", "query_object_valid"):
        batch[name] = torch.from_numpy(np.stack([s[name] for s in samples]))
    for name in ("candidate_node", "candidate_edge", "candidate_object_node", "candidate_object_edge",
                 "candidate_object_membership", "candidate_object_valid"):
        if name in samples[0]: batch[name] = torch.from_numpy(np.stack([s[name] for s in samples]))
    for name in ("label_op", "label_source", "label_target", "label_transform", "label_dr", "label_dc"):
        batch[name] = torch.tensor([s[name] for s in samples], dtype=torch.long)
    if "label_candidate" in samples[0]:
        batch["label_candidate"] = torch.from_numpy(np.stack([s["label_candidate"] for s in samples]))
    return batch


def _place(grid, cells, color):
    for r, c in cells: grid[r, c] = color


def _translate_cells(cells, dr, dc): return [(r+dr, c+dc) for r, c in cells]


def _toward_delta(source, target):
    sr, sc = np.mean(source, 0); tr, tc = np.mean(target, 0)
    if abs(tr-sr) >= abs(tc-sc): return (1 if tr > sr else -1, 0)
    return (0, 1 if tc > sc else -1)


SHAPE_LIBRARY = (
    np.array([[1, 1], [1, 0]], bool),                         # L
    np.array([[1, 1, 1], [0, 1, 0]], bool),                  # T
    np.array([[1, 1, 1]], bool),                             # line
    np.array([[1, 1], [1, 1]], bool),                        # square
    np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], bool),      # plus
    np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], bool),      # hollow square
    np.array([[1, 0, 0], [1, 1, 1]], bool),                  # long L
    np.array([[1, 1, 0], [0, 1, 1]], bool),                  # zigzag
    np.array([[0, 1, 1, 1], [1, 1, 0, 1],
              [1, 0, 0, 1], [1, 1, 1, 1]], bool),           # notched ring
)
ASYMMETRIC_SHAPES = (0, 1, 6, 7)
TRANSFORM_CODES = {"rotate90": 1, "rotate180": 2, "reflect_left_right": 4, "reflect_up_down": 6}


def _nontrivial_shape_ids(transform: int) -> tuple[int, ...]:
    return tuple(i for i, mask in enumerate(SHAPE_LIBRARY)
                 if (z := _d4(mask, transform)).shape != mask.shape or not np.array_equal(z, mask))


def _outline_mask(mask: np.ndarray) -> np.ndarray:
    out = np.zeros_like(mask, dtype=bool)
    out[0, :] = True; out[-1, :] = True; out[:, 0] = True; out[:, -1] = True
    return out


def _source_shape_pool(op_name: str) -> tuple[int, ...]:
    if op_name in TRANSFORM_CODES: return _nontrivial_shape_ids(TRANSFORM_CODES[op_name])
    if op_name == "fill_bbox": return tuple(i for i, m in enumerate(SHAPE_LIBRARY) if int(m.sum()) < m.size)
    if op_name == "outline_bbox": return tuple(i for i, m in enumerate(SHAPE_LIBRARY) if not np.array_equal(m, _outline_mask(m)))
    if op_name == "fill_holes": return (8,)
    return tuple(range(len(SHAPE_LIBRARY)))


def _random_d4(mask: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    return _d4(mask, int(rng.integers(8))).copy()


def _place_random_mask(grid: np.ndarray, mask: np.ndarray, color: int, rng: np.random.Generator,
                       occupied: np.ndarray, margin: int = 1, attempts: int = 100) -> list[tuple[int, int]] | None:
    h, w = grid.shape; mh, mw = mask.shape
    if mh + 2*margin > h or mw + 2*margin > w: return None
    for _ in range(attempts):
        r0 = int(rng.integers(margin, h - mh - margin + 1)); c0 = int(rng.integers(margin, w - mw - margin + 1))
        cells = [(r0+int(r), c0+int(c)) for r, c in np.argwhere(mask)]
        halo = [(r+dr, c+dc) for r, c in cells for dr in (-1, 0, 1) for dc in (-1, 0, 1)
                if 0 <= r+dr < h and 0 <= c+dc < w]
        if any(occupied[r, c] for r, c in halo): continue
        _place(grid, cells, color)
        for r, c in cells: occupied[r, c] = True
        return cells
    return None


def _program_aux_labels(op_name: str, source, target) -> tuple[int, int, int]:
    transform = {"rotate90": 1, "rotate180": 2, "reflect_left_right": 4, "reflect_up_down": 6}.get(op_name, 0)
    dr = dc = 0
    if op_name in ("move_toward", "copy_toward", "move_until_touch"):
        dr, dc = _toward_delta(source, target)
    elif op_name in ("move_away", "move_away_until_border"):
        tr, tc = _toward_delta(source, target); dr, dc = -tr, -tc
    elif op_name == "align_row": dr = min(r for r, _ in target) - min(r for r, _ in source)
    elif op_name == "align_column": dc = min(c for _, c in target) - min(c for _, c in source)
    return transform, int(np.clip(dr, -4, 4)) + 4, int(np.clip(dc, -4, 4)) + 4


def _synthetic_pair(rng: np.random.Generator, op_name: str, source_shape_id: int,
                    target_shape_id: int, h: int = 12, w: int = 12):
    for _ in range(200):
        background_color = int(rng.integers(10))
        colors = rng.choice([c for c in range(10) if c != background_color], size=4, replace=False)
        source_color, target_color = map(int, colors[:2])
        x = np.full((h, w), background_color, np.int8); occupied = np.zeros((h, w), bool)
        source_mask = _random_d4(SHAPE_LIBRARY[source_shape_id], rng)
        target_mask = _random_d4(SHAPE_LIBRARY[target_shape_id], rng)
        source = _place_random_mask(x, source_mask, source_color, rng, occupied)
        target = _place_random_mask(x, target_mask, target_color, rng, occupied)
        if source is None or target is None: continue
        # One or two distractor layers make source/context selection relational.
        for distractor_color in map(int, colors[2:2+int(rng.integers(1, 3))]):
            shape = _random_d4(SHAPE_LIBRARY[int(rng.integers(len(SHAPE_LIBRARY)))], rng)
            _place_random_mask(x, shape, distractor_color, rng, occupied, attempts=30)
        program = {"op": op_name, "source": source_color, "target": target_color, "dr": 0, "dc": 0}
        y = execute_relation_program(x, program)
        if y is None: continue
        if op_name != "identity" and np.array_equal(x, y): continue
        transform, dr, dc = _program_aux_labels(op_name, source, target)
        return x, y, {"source": source_color, "target": target_color, "transform": transform, "dr": dr, "dc": dc}
    raise RuntimeError(f"could not generate valid synthetic pair for {op_name}")


def _operator_is_identifiable(pairs, expected_op: str) -> bool:
    """Reject demonstrations that are exactly explained by another DSL op."""
    for alternative in PROGRAM_OPS:
        if alternative == expected_op:
            continue
        matches_every_demo = True
        for x, y, label in pairs:
            candidate = execute_relation_program(x, {
                "op": alternative, "source": label["source"], "target": label["target"],
                "transform": label["transform"], "dr": label["dr"] - 4, "dc": label["dc"] - 4,
            })
            if candidate is None or not np.array_equal(candidate, y):
                matches_every_demo = False
                break
        if matches_every_demo:
            return False
    return True


def _negative_candidate(grid: np.ndarray, correct: np.ndarray, label: dict,
                        expected_op: str, rng: np.random.Generator) -> np.ndarray:
    present = list(map(int, np.unique(grid))); proposals = []
    alternatives = [name for name in PROGRAM_OPS if name != expected_op]
    rng.shuffle(alternatives)
    for op in alternatives:
        role_pairs = [(label["source"], label["target"])]
        role_pairs += [(int(rng.choice(present)), int(rng.choice(present))) for _ in range(3)]
        for source, target in role_pairs:
            if op in TARGET_REQUIRED_OPS and source == target: continue
            candidate = execute_relation_program(grid, {"op": op, "source": source, "target": target,
                "transform": label["transform"], "dr": label["dr"]-4, "dc": label["dc"]-4})
            if candidate is not None and candidate.shape == correct.shape and not np.array_equal(candidate, correct):
                proposals.append(candidate)
        if proposals and rng.random() < .35: break
    return proposals[int(rng.integers(len(proposals)))] if proposals else grid.copy()


class SyntheticRelationDataset(Dataset):
    """Program-labelled relational tasks with fresh colour permutations."""
    def __init__(self, size: int = 4096, demos: int = 2, seed: int = 0, cache: bool = False):
        self.size, self.demos, self.seed, self.cache = size, demos, seed, cache
        self._cache: dict[int, dict] = {}
    def __len__(self): return self.size
    def __getitem__(self, index):
        if index in self._cache:
            return self._cache[index]
        rng = np.random.default_rng(self.seed + index); op_id = int(rng.integers(len(PROGRAM_OPS))); op_name = PROGRAM_OPS[op_id]
        for _ in range(100):
            source_shape_id = int(rng.choice(_source_shape_pool(op_name)))
            target_shape_id = int(rng.integers(len(SHAPE_LIBRARY)))
            demo_pairs = [_synthetic_pair(rng, op_name, source_shape_id, target_shape_id) for _ in range(self.demos)]
            if _operator_is_identifiable(demo_pairs, op_name):
                pairs = demo_pairs + [_synthetic_pair(rng, op_name, source_shape_id, target_shape_id)]
                break
        else:
            raise RuntimeError(f"could not generate identifiable synthetic task for {op_name}")
        task = {"train": [{"input": x.tolist(), "output": y.tolist()} for x, y, _ in pairs[:-1]],
                "test": [{"input": pairs[-1][0].tolist()}]}
        sample = encode_task(task); label = pairs[-1][2]
        sample.update({"label_op": op_id, "label_source": label["source"], "label_target": label["target"],
                       "label_transform": label["transform"], "label_dr": label["dr"], "label_dc": label["dc"]})
        candidate_grids = [pairs[-1][1], _negative_candidate(pairs[-1][0], pairs[-1][1], label, op_name, rng)]
        candidate_layer = [extract_layer_graph(grid) for grid in candidate_grids]
        candidate_object = [extract_object_graph(grid) for grid in candidate_grids]
        sample.update({
            "candidate_node": np.stack([z[0] for z in candidate_layer]),
            "candidate_edge": np.stack([z[1] for z in candidate_layer]),
            "candidate_object_node": np.stack([z[0] for z in candidate_object]),
            "candidate_object_edge": np.stack([z[1] for z in candidate_object]),
            "candidate_object_membership": np.stack([z[2] for z in candidate_object]),
            "candidate_object_valid": np.stack([z[3] for z in candidate_object]),
            "label_candidate": np.asarray([1., 0.], np.float32),
        })
        if self.cache:
            self._cache[index] = sample
        return sample


COMPOSITION_TEMPLATES = ("keep_fill", "delete_smallest", "recolor_largest",
                         "crop_largest", "transform_largest", "gravity_smallest")


def _synthetic_composition_pair(rng: np.random.Generator, template: str,
                                direction: str, transform: int):
    for _ in range(200):
        bg = int(rng.integers(10)); colours = rng.choice([c for c in range(10) if c != bg], 2, replace=False)
        source, target = map(int, colours); grid = np.full((12, 12), bg, np.int8); occupied = np.zeros_like(grid, bool)
        masks = (np.ones((1, 1), bool), _random_d4(SHAPE_LIBRARY[0], rng), _random_d4(SHAPE_LIBRARY[6], rng))
        if any(_place_random_mask(grid, mask, source, rng, occupied) is None for mask in masks): continue
        _place_random_mask(grid, _random_d4(SHAPE_LIBRARY[1], rng), target, rng, occupied, attempts=40)
        if template == "keep_fill":
            steps = [{"op": "keep_selected", "source": source, "selector": "largest"},
                     {"op": "fill_selected_bbox", "source": source, "selector": "all"}]
        elif template == "delete_smallest":
            steps = [{"op": "delete_selected", "source": source, "selector": "smallest"}]
        elif template == "recolor_largest":
            steps = [{"op": "recolor_selected", "source": source, "target": target, "selector": "largest"}]
        elif template == "crop_largest":
            steps = [{"op": "crop_selected", "source": source, "selector": "largest"}]
        elif template == "transform_largest":
            steps = [{"op": "transform_selected", "source": source, "selector": "largest", "transform": transform}]
        else:
            steps = [{"op": "gravity_selected", "source": source, "selector": "smallest", "direction": direction}]
        output = execute_compositional_program(grid, {"steps": steps})
        if output is None or np.array_equal(output, grid): continue
        if len(steps) == 2:
            negative = execute_compositional_program(grid, {"steps": steps[:1]})
        else:
            wrong = dict(steps[0]); wrong["selector"] = "smallest" if wrong["selector"] == "largest" else "largest"
            negative = execute_compositional_program(grid, {"steps": [wrong]})
        if negative is None or negative.shape != output.shape or np.array_equal(negative, output):
            negative = grid.copy() if grid.shape == output.shape else np.full_like(output, bg)
        return grid, output, negative, source, target
    raise RuntimeError(f"could not generate compositional pair for {template}")


class SyntheticCompositionCriticDataset(Dataset):
    """Two-stage object tasks used only to train candidate consistency."""
    def __init__(self, size: int = 1024, demos: int = 2, seed: int = 0, cache: bool = False):
        self.size, self.demos, self.seed, self.cache = size, demos, seed, cache; self._cache = {}
    def __len__(self): return self.size
    def __getitem__(self, index):
        if index in self._cache: return self._cache[index]
        rng = np.random.default_rng(self.seed+index); template = COMPOSITION_TEMPLATES[index % len(COMPOSITION_TEMPLATES)]
        direction = ("down", "up", "left", "right")[int(rng.integers(4))]
        transform = (1, 2, 4, 6)[int(rng.integers(4))]
        pairs = [_synthetic_composition_pair(rng, template, direction, transform) for _ in range(self.demos+1)]
        task = {"train": [{"input": x.tolist(), "output": y.tolist()} for x, y, _, _, _ in pairs[:-1]],
                "test": [{"input": pairs[-1][0].tolist()}]}
        sample = encode_task(task); x, positive, negative, source, target = pairs[-1]
        layer_graphs = [extract_layer_graph(z) for z in (positive, negative)]
        object_graphs = [extract_object_graph(z) for z in (positive, negative)]
        sample.update({"label_op": 0, "label_source": source, "label_target": target,
                       "label_transform": 0, "label_dr": 4, "label_dc": 4,
                       "candidate_node": np.stack([z[0] for z in layer_graphs]),
                       "candidate_edge": np.stack([z[1] for z in layer_graphs]),
                       "candidate_object_node": np.stack([z[0] for z in object_graphs]),
                       "candidate_object_edge": np.stack([z[1] for z in object_graphs]),
                       "candidate_object_membership": np.stack([z[2] for z in object_graphs]),
                       "candidate_object_valid": np.stack([z[3] for z in object_graphs]),
                       "label_candidate": np.asarray([1., 0.], np.float32)})
        if self.cache: self._cache[index] = sample
        return sample


def candidate_consistency_loss(outputs: dict, batch: dict) -> torch.Tensor:
    return F.binary_cross_entropy_with_logits(outputs["candidate"], batch["label_candidate"])


def _operation_mask(batch: dict, operation_names) -> torch.Tensor:
    ids = torch.tensor([PROGRAM_OPS.index(name) for name in operation_names],
                       device=batch["label_op"].device, dtype=batch["label_op"].dtype)
    if not ids.numel():
        return torch.zeros_like(batch["label_op"], dtype=torch.bool)
    return (batch["label_op"][:, None] == ids[None]).any(-1)


def _masked_cross_entropy(logits: torch.Tensor, labels: torch.Tensor, mask: torch.Tensor,
                          class_weight: torch.Tensor | None = None) -> torch.Tensor:
    if bool(mask.any()):
        return F.cross_entropy(logits[mask], labels[mask], weight=class_weight)
    return logits.sum() * 0


def relation_program_loss(outputs: dict, batch: dict) -> torch.Tensor:
    """Semantic loss: supervise only arguments actually consumed by an operation."""
    loss = F.cross_entropy(outputs["op"], batch["label_op"])
    swap_mask = _operation_mask(batch, {"swap_source_context_colors"})
    source_mask = _operation_mask(batch, set(PROGRAM_OPS) - {"identity", "swap_source_context_colors"})
    target_mask = _operation_mask(batch, TARGET_REQUIRED_OPS - {"swap_source_context_colors"})
    transform_mask = _operation_mask(batch, TRANSFORM_REQUIRED_OPS)
    delta_mask = _operation_mask(batch, DELTA_REQUIRED_OPS)
    loss = loss + .5 * _masked_cross_entropy(outputs["source"], batch["label_source"], source_mask)
    loss = loss + .5 * _masked_cross_entropy(outputs["target"], batch["label_target"], target_mask)
    loss = loss + .25 * _masked_cross_entropy(outputs["transform"], batch["label_transform"], transform_mask)
    delta_class_weight = torch.ones(9, device=outputs["dr"].device); delta_class_weight[4] = .25
    loss = loss + .25 * _masked_cross_entropy(outputs["dr"], batch["label_dr"], delta_mask, delta_class_weight)
    loss = loss + .25 * _masked_cross_entropy(outputs["dc"], batch["label_dc"], delta_mask, delta_class_weight)
    if bool(swap_mask.any()):
        source, target = outputs["source"][swap_mask], outputs["target"][swap_mask]
        source_label, target_label = batch["label_source"][swap_mask], batch["label_target"][swap_mask]
        direct = F.cross_entropy(source, source_label, reduction="none") + F.cross_entropy(target, target_label, reduction="none")
        reversed_pair = F.cross_entropy(source, target_label, reduction="none") + F.cross_entropy(target, source_label, reduction="none")
        loss = loss + .5 * torch.minimum(direct, reversed_pair).mean()
    if "candidate" in outputs and "label_candidate" in batch:
        loss = loss + .5 * F.binary_cross_entropy_with_logits(outputs["candidate"], batch["label_candidate"])
    return loss


@torch.no_grad()
def relation_program_accuracy(outputs: dict, batch: dict) -> dict[str, float]:
    def masked_accuracy(name: str, mask: torch.Tensor) -> float:
        if not bool(mask.any()):
            return float("nan")
        return float((outputs[name].argmax(-1)[mask] == batch[f"label_{name}"][mask]).float().mean())

    swap_mask = _operation_mask(batch, {"swap_source_context_colors"})
    source_mask = _operation_mask(batch, set(PROGRAM_OPS) - {"identity", "swap_source_context_colors"})
    target_mask = _operation_mask(batch, TARGET_REQUIRED_OPS - {"swap_source_context_colors"})
    transform_mask = _operation_mask(batch, TRANSFORM_REQUIRED_OPS)
    delta_mask = _operation_mask(batch, DELTA_REQUIRED_OPS)
    result = {
        "op": float((outputs["op"].argmax(-1) == batch["label_op"]).float().mean()),
        "source": masked_accuracy("source", source_mask),
        "target": masked_accuracy("target", target_mask),
        "transform": masked_accuracy("transform", transform_mask),
        "dr": masked_accuracy("dr", delta_mask),
        "dc": masked_accuracy("dc", delta_mask),
    }
    if bool(swap_mask.any()):
        source = outputs["source"].argmax(-1)[swap_mask]
        target = outputs["target"].argmax(-1)[swap_mask]
        source_label = batch["label_source"][swap_mask]
        target_label = batch["label_target"][swap_mask]
        pair_ok = ((source == source_label) & (target == target_label)) | ((source == target_label) & (target == source_label))
        result["swap_pair"] = float(pair_ok.float().mean())
    else:
        result["swap_pair"] = float("nan")
    if "candidate" in outputs and "label_candidate" in batch:
        predicted = outputs["candidate"] > 0
        result["candidate"] = float((predicted == batch["label_candidate"].bool()).float().mean())
        result["candidate_rank"] = float((outputs["candidate"][:, 0] > outputs["candidate"][:, 1]).float().mean())
    return result


def move_batch(batch: dict, device: torch.device) -> dict:
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def forward_relation_batch(model: nn.Module, batch: dict) -> dict:
    return model(
        batch["demo_in_node"], batch["demo_in_edge"], batch["demo_out_node"], batch["demo_out_edge"],
        batch["query_node"], batch["query_edge"], batch.get("demo_mask"),
        demo_in_object_node=batch.get("demo_in_object_node"),
        demo_in_object_edge=batch.get("demo_in_object_edge"),
        demo_in_object_membership=batch.get("demo_in_object_membership"),
        demo_in_object_valid=batch.get("demo_in_object_valid"),
        demo_out_object_node=batch.get("demo_out_object_node"),
        demo_out_object_edge=batch.get("demo_out_object_edge"),
        demo_out_object_membership=batch.get("demo_out_object_membership"),
        demo_out_object_valid=batch.get("demo_out_object_valid"),
        query_object_node=batch.get("query_object_node"),
        query_object_edge=batch.get("query_object_edge"),
        query_object_membership=batch.get("query_object_membership"),
        query_object_valid=batch.get("query_object_valid"),
        candidate_node=batch.get("candidate_node"),
        candidate_edge=batch.get("candidate_edge"),
        candidate_object_node=batch.get("candidate_object_node"),
        candidate_object_edge=batch.get("candidate_object_edge"),
        candidate_object_membership=batch.get("candidate_object_membership"),
        candidate_object_valid=batch.get("candidate_object_valid"),
    )


@torch.no_grad()
def propose_relation_programs(model: nn.Module, task: dict, device: torch.device, topk: int = 5) -> list[dict]:
    encoded = encode_task(task); batch = collate_relation_batch([{**encoded,
        "label_op": 0, "label_source": 0, "label_target": 0, "label_transform": 0, "label_dr": 4, "label_dc": 4}])
    batch = move_batch(batch, device); model.eval()
    out = forward_relation_batch(model, batch)
    tops = {k: torch.topk(v.softmax(-1)[0], min(topk, v.shape[-1])) for k, v in out.items()}
    candidates = []
    for oi in range(min(5, len(tops["op"].indices))):
        op_name = PROGRAM_OPS[int(tops["op"].indices[oi])]
        source_options = [(0, torch.ones_like(tops["op"].values[oi]))] if op_name == "identity" else [
            (int(tops["source"].indices[si]), tops["source"].values[si])
            for si in range(min(3, len(tops["source"].indices)))]
        target_options = [(0, torch.ones_like(tops["op"].values[oi]))] if op_name not in TARGET_REQUIRED_OPS else [
            (int(tops["target"].indices[ti]), tops["target"].values[ti])
            for ti in range(min(3, len(tops["target"].indices)))]
        for source, source_score in source_options:
            for target, target_score in target_options:
                if op_name in TARGET_REQUIRED_OPS and source == target:
                    continue
                score = float(tops["op"].values[oi] * source_score * target_score)
                candidates.append({"op": op_name, "source": source, "target": target,
                    "transform": int(tops["transform"].indices[0]), "dr": int(tops["dr"].indices[0])-4,
                    "dc": int(tops["dc"].indices[0])-4, "score": score})
    return sorted(candidates, key=lambda z: -z["score"])[:topk]


@torch.no_grad()
def score_relation_candidates(model: nn.Module, task: dict, candidates: list[Any],
                              device: torch.device, query_index: int = 0) -> list[float]:
    """Score arbitrary LLM/symbolic outputs by demonstration relation consistency."""
    if not candidates: return []
    encoded = encode_task(task, query_index)
    layer_graphs = [extract_layer_graph(candidate) for candidate in candidates]
    object_graphs = [extract_object_graph(candidate) for candidate in candidates]
    encoded.update({
        "candidate_node": np.stack([z[0] for z in layer_graphs]),
        "candidate_edge": np.stack([z[1] for z in layer_graphs]),
        "candidate_object_node": np.stack([z[0] for z in object_graphs]),
        "candidate_object_edge": np.stack([z[1] for z in object_graphs]),
        "candidate_object_membership": np.stack([z[2] for z in object_graphs]),
        "candidate_object_valid": np.stack([z[3] for z in object_graphs]),
        "label_candidate": np.zeros(len(candidates), np.float32),
        "label_op": 0, "label_source": 0, "label_target": 0,
        "label_transform": 0, "label_dr": 4, "label_dc": 4,
    })
    batch = move_batch(collate_relation_batch([encoded]), device); model.eval()
    scores = forward_relation_batch(model, batch)["candidate"].sigmoid()[0]
    return [float(value) for value in scores]


def _infer_background(g):
    g = np.asarray(g); border = np.concatenate((g[0], g[-1], g[:, 0], g[:, -1])); counts = np.bincount(border, minlength=10)
    return int(np.argmax(counts))


def _layer_role_statistics(grid: Any) -> dict[int, dict]:
    g = np.asarray(grid, dtype=np.int8); bg = _infer_background(g); stats = {}
    for colour in sorted(set(map(int, g.ravel())) - {bg}):
        mask = g == colour; components = _components(mask); component_areas = [len(z) for z in components]
        holes = 0; touches = False
        for cells in components:
            component_mask = np.zeros_like(mask)
            for r, c in cells: component_mask[r, c] = True
            crop = _crop(component_mask); holes += _holes(crop)
            r0, r1, c0, c1 = _bbox(component_mask)
            touches |= r0 == 0 or r1 == g.shape[0]-1 or c0 == 0 or c1 == g.shape[1]-1
        stats[colour] = {"area": int(mask.sum()), "components": len(components),
                         "smallest": min(component_areas), "largest": max(component_areas),
                         "holes": holes, "touches": touches}
    return stats


def resolve_layer_role(grid: Any, role: str) -> int | None:
    """Resolve a categorical role only when it identifies exactly one layer."""
    stats = _layer_role_statistics(grid)
    if role not in LAYER_ROLES or not stats: return None
    if role == "only_foreground": candidates = list(stats) if len(stats) == 1 else []
    elif role == "has_hole": candidates = [c for c, z in stats.items() if z["holes"] > 0]
    elif role == "border_touching": candidates = [c for c, z in stats.items() if z["touches"]]
    elif role == "interior_only": candidates = [c for c, z in stats.items() if not z["touches"]]
    else:
        field, choose = {
            "rarest": ("area", min), "most_frequent": ("area", max),
            "fewest_components": ("components", min), "most_components": ("components", max),
            "smallest_object_layer": ("smallest", min), "largest_object_layer": ("largest", max),
        }[role]
        target = choose(z[field] for z in stats.values())
        candidates = [c for c, z in stats.items() if z[field] == target]
    return int(candidates[0]) if len(candidates) == 1 else None


def layer_role_candidates(grid: Any, colour: int, include_value: bool = True) -> list[dict]:
    result = [{"role": role} for role in LAYER_ROLES if resolve_layer_role(grid, role) == int(colour)]
    if include_value: result.append({"value": int(colour)})
    return result


def _resolve_program_colour(grid: Any, program: dict, name: str) -> tuple[int | None, bool]:
    role_key = f"{name}_role"
    if role_key in program:
        colour = resolve_layer_role(grid, str(program[role_key])); return colour, colour is not None
    if name in program and program[name] is not None:
        colour = int(program[name]); return (colour, 0 <= colour <= 9)
    return None, True


def _touches_cells(a, b) -> bool:
    bset = set(b)
    return any((r+dr, c+dc) in bset for r, c in a for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)))


def _inside(cells, shape) -> bool:
    return all(0 <= r < shape[0] and 0 <= c < shape[1] for r, c in cells)


def _move_until(source_cells, target_cells, dr, dc, shape, until_touch=False):
    current = list(source_cells)
    for _ in range(max(shape) * 2):
        if until_touch and _touches_cells(current, target_cells): break
        nxt = _translate_cells(current, dr, dc)
        if not _inside(nxt, shape) or any(p in set(target_cells) for p in nxt): break
        current = nxt
    return current


def _fill_binary_holes(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask, bool); h, w = mask.shape; outside = np.zeros_like(mask); stack = []
    for r in range(h):
        for c in range(w):
            if (r in (0, h-1) or c in (0, w-1)) and not mask[r, c] and not outside[r, c]:
                outside[r, c] = True; stack.append((r, c))
    while stack:
        r, c = stack.pop()
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nr, nc = r+dr, c+dc
            if 0 <= nr < h and 0 <= nc < w and not mask[nr, nc] and not outside[nr, nc]:
                outside[nr, nc] = True; stack.append((nr, nc))
    return mask | (~mask & ~outside)


def _transform_source(out, g, source, bg, transform):
    src = np.argwhere(g == source)
    if not len(src): return None
    r0, c0 = src.min(0); r1, c1 = src.max(0); mask = g[r0:r1+1, c0:c1+1] == source
    z = _d4(mask, transform).astype(bool); out[g == source] = bg
    if r0+z.shape[0] > g.shape[0] or c0+z.shape[1] > g.shape[1]: return None
    view = out[r0:r0+z.shape[0], c0:c0+z.shape[1]]
    if np.any(z & (view != bg)): return None
    view[z] = source
    return out


def execute_relation_program(grid: Any, program: dict) -> np.ndarray | None:
    """Execute the expanded relation DSL deterministically."""
    g = np.asarray(grid, dtype=np.int8); out = g.copy(); bg = _infer_background(g); op = program["op"]
    if op == "identity": return out
    source, source_ok = _resolve_program_colour(g, program, "source")
    target, target_ok = _resolve_program_colour(g, program, "target")
    if not source_ok or source is None or (op in TARGET_REQUIRED_OPS and (not target_ok or target is None)):
        return None
    target = 0 if target is None else target
    src, tgt = np.argwhere(g == source), np.argwhere(g == target)
    if not len(src): return None
    source_cells = [tuple(map(int, p)) for p in src]
    if op == "recolor_as_context":
        if not len(tgt): return None
        out[g == source] = target; return out
    if op == "swap_source_context_colors":
        if not len(tgt): return None
        source_mask, target_mask = g == source, g == target
        out[source_mask], out[target_mask] = target, source
        return out
    if op == "delete_source": out[g == source] = bg; return out
    if op == "fill_bbox":
        r0, c0 = src.min(0); r1, c1 = src.max(0); out[r0:r1+1, c0:c1+1] = source; return out
    if op == "outline_bbox":
        r0, c0 = src.min(0); r1, c1 = src.max(0); out[g == source] = bg
        out[r0, c0:c1+1] = source; out[r1, c0:c1+1] = source
        out[r0:r1+1, c0] = source; out[r0:r1+1, c1] = source
        return out
    if op == "fill_holes":
        r0, c0 = src.min(0); r1, c1 = src.max(0); mask = g[r0:r1+1, c0:c1+1] == source
        view = out[r0:r1+1, c0:c1+1]; view[_fill_binary_holes(mask)] = source
        return out
    transform_codes = {"rotate90": 1, "rotate180": 2, "reflect_left_right": 4, "reflect_up_down": 6}
    if op in transform_codes: return _transform_source(out, g, source, bg, transform_codes[op])
    if not len(tgt): return None
    target_cells = [tuple(map(int, p)) for p in tgt]
    if op in ("move_toward", "copy_toward", "move_until_touch"):
        dr, dc = _toward_delta(source_cells, target_cells)
    elif op in ("move_away", "move_away_until_border"):
        tr, tc = _toward_delta(source_cells, target_cells); dr, dc = -tr, -tc
    elif op == "align_row": dr, dc = int(tgt[:,0].min()-src[:,0].min()), 0
    elif op == "align_column": dr, dc = 0, int(tgt[:,1].min()-src[:,1].min())
    else: dr, dc = int(program.get("dr", 0)), int(program.get("dc", 0))
    if op == "move_until_touch": moved = _move_until(source_cells, target_cells, dr, dc, g.shape, True)
    elif op == "move_away_until_border": moved = _move_until(source_cells, target_cells, dr, dc, g.shape, False)
    else: moved = _translate_cells(source_cells, dr, dc)
    if any(not (0 <= r < g.shape[0] and 0 <= c < g.shape[1]) for r, c in moved): return None
    if op == "connect_source_to_context":
        a, b = min(((tuple(map(int, p)), tuple(map(int, q))) for p in src for q in tgt), key=lambda z: abs(z[0][0]-z[1][0])+abs(z[0][1]-z[1][1]))
        r, c = a; tr, tc = b
        while r != tr:
            r += 1 if tr > r else -1
            if (r, c) != b: out[r, c] = source
        while c != tc:
            c += 1 if tc > c else -1
            if (r, c) != b: out[r, c] = source
        return out
    if op != "copy_toward": out[g == source] = bg
    # Do not erase/overwrite unrelated context in synthetic or real execution.
    if any(out[r, c] not in (bg, source) for r, c in moved): return None
    _place(out, moved, source); return out


def _component_records(grid: np.ndarray, source: int | None = None) -> list[dict]:
    bg = _infer_background(grid); records = []
    colours = [source] if source is not None else sorted(set(map(int, grid.ravel())) - {bg})
    for colour in colours:
        if colour is None or colour == bg:
            continue
        for cells in _components(grid == colour):
            mask = np.zeros_like(grid, dtype=bool)
            for r, c in cells: mask[r, c] = True
            box = _bbox(mask); crop = _crop(mask)
            records.append({"colour": int(colour), "cells": cells, "mask": mask, "box": box,
                            "area": len(cells), "shape": _canonical_shape_bitmap(mask).tobytes(),
                            "holes": _holes(crop)})
    return records


def _select_component_records(records: list[dict], selector: str, shape: tuple[int, int]) -> list[dict]:
    if selector not in OBJECT_SELECTORS or not records:
        return []
    if selector == "all": return records
    if selector == "largest": value = max(z["area"] for z in records); return [z for z in records if z["area"] == value]
    if selector == "smallest": value = min(z["area"] for z in records); return [z for z in records if z["area"] == value]
    if selector == "topmost": value = min(z["box"][0] for z in records); return [z for z in records if z["box"][0] == value]
    if selector == "bottommost": value = max(z["box"][1] for z in records); return [z for z in records if z["box"][1] == value]
    if selector == "leftmost": value = min(z["box"][2] for z in records); return [z for z in records if z["box"][2] == value]
    if selector == "rightmost": value = max(z["box"][3] for z in records); return [z for z in records if z["box"][3] == value]
    if selector == "border":
        h, w = shape; return [z for z in records if z["box"][0] == 0 or z["box"][1] == h-1 or z["box"][2] == 0 or z["box"][3] == w-1]
    if selector == "interior":
        border = set(id(z) for z in _select_component_records(records, "border", shape)); return [z for z in records if id(z) not in border]
    if selector == "has_hole": return [z for z in records if z["holes"] > 0]
    key = "area" if selector.endswith("area") else "shape"
    counts = Counter(z[key] for z in records); unique = selector.startswith("unique")
    return [z for z in records if (counts[z[key]] == 1) == unique]


def _transform_component(out: np.ndarray, original: np.ndarray, record: dict, transform: int, bg: int) -> bool:
    r0, r1, c0, c1 = record["box"]; mask = _crop(record["mask"])
    transformed = _d4(mask, transform).astype(bool)
    for r, c in record["cells"]: out[r, c] = bg
    if r0 + transformed.shape[0] > out.shape[0] or c0 + transformed.shape[1] > out.shape[1]:
        return False
    view = out[r0:r0+transformed.shape[0], c0:c0+transformed.shape[1]]
    if np.any(transformed & (view != bg)):
        return False
    view[transformed] = record["colour"]
    return True


def execute_object_step(grid: Any, step: dict) -> np.ndarray | None:
    """Execute a component-selective primitive while preserving other layers."""
    g = np.asarray(grid, dtype=np.int8); op = step.get("op")
    if op not in OBJECT_STEP_OPS:
        return None
    source, source_ok = _resolve_program_colour(g, step, "source")
    if not source_ok: return None
    selector = step.get("selector", "all"); records = _component_records(g, source)
    selected = _select_component_records(records, selector, g.shape)
    if not selected:
        return None
    bg = _infer_background(g); out = g.copy()
    selected_cells = {cell for record in selected for cell in record["cells"]}
    if op == "keep_selected":
        out = np.full_like(g, bg)
        for r, c in selected_cells: out[r, c] = g[r, c]
    elif op == "delete_selected":
        for r, c in selected_cells: out[r, c] = bg
    elif op == "recolor_selected":
        target, target_ok = _resolve_program_colour(g, step, "target")
        if not target_ok or target is None: return None
        for r, c in selected_cells: out[r, c] = target
    elif op == "crop_selected":
        canvas = np.full_like(g, bg)
        for r, c in selected_cells: canvas[r, c] = g[r, c]
        box = _bbox(canvas != bg)
        if box is None: return None
        r0, r1, c0, c1 = box; out = canvas[r0:r1+1, c0:c1+1]
    elif op in ("fill_selected_bbox", "outline_selected_bbox"):
        for record in selected:
            for r, c in record["cells"]: out[r, c] = bg
            r0, r1, c0, c1 = record["box"]; colour = record["colour"]
            if op == "fill_selected_bbox": out[r0:r1+1, c0:c1+1] = colour
            else:
                out[r0, c0:c1+1] = colour; out[r1, c0:c1+1] = colour
                out[r0:r1+1, c0] = colour; out[r0:r1+1, c1] = colour
    elif op == "transform_selected":
        transform = int(step.get("transform", 0))
        if not 0 <= transform < 8: return None
        for r, c in selected_cells: out[r, c] = bg
        for record in selected:
            if not _transform_component(out, g, record, transform, bg): return None
    elif op == "gravity_selected":
        direction = step.get("direction", "down")
        if direction not in {"down", "up", "left", "right"}: return None
        dr, dc = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}[direction]
        fixed = {(r, c) for r in range(g.shape[0]) for c in range(g.shape[1])
                 if g[r, c] != bg and (r, c) not in selected_cells}
        moved = set(selected_cells)
        while True:
            nxt = {(r+dr, c+dc) for r, c in moved}
            if any(not (0 <= r < g.shape[0] and 0 <= c < g.shape[1]) or (r, c) in fixed for r, c in nxt): break
            moved = nxt
        for r, c in selected_cells: out[r, c] = bg
        # A selected set may contain multiple colours; preserve each by the common displacement.
        shift_r = min(r for r, _ in moved) - min(r for r, _ in selected_cells)
        shift_c = min(c for _, c in moved) - min(c for _, c in selected_cells)
        for r, c in selected_cells: out[r+shift_r, c+shift_c] = g[r, c]
    return out if out.ndim == 2 and 0 < out.shape[0] <= 30 and 0 < out.shape[1] <= 30 else None


def execute_compositional_program(grid: Any, program: dict) -> np.ndarray | None:
    """Execute one or two typed steps; every intermediate grid must be valid."""
    steps = program.get("steps")
    if not isinstance(steps, (list, tuple)) or not 1 <= len(steps) <= 2:
        return None
    current = np.asarray(grid, dtype=np.int8)
    for step in steps:
        if step.get("op") in PROGRAM_OPS:
            current = execute_relation_program(current, step)
        else:
            current = execute_object_step(current, step)
        if current is None:
            return None
    return current


def verify_compositional_program(task: dict, program: dict) -> bool:
    return all((prediction := execute_compositional_program(pair["input"], program)) is not None
               and np.array_equal(prediction, np.asarray(pair["output"])) for pair in task["train"])


def enumerate_object_steps(grid: Any, target: Any | None = None) -> list[dict]:
    """Generate routed object primitives without using numeric colour order."""
    g = np.asarray(grid, dtype=np.int8); bg = _infer_background(g)
    colours = list(set(map(int, g.ravel())) - {bg})
    colours.sort(key=lambda colour: tuple(np.argwhere(g == colour).min(0)))
    target_array = np.asarray(target) if target is not None else None
    target_bg = _infer_background(target_array) if target_array is not None else None
    if target_array is not None:
        target_colours = list(set(map(int, target_array.ravel())) - {target_bg})
        target_colours.sort(key=lambda colour: tuple(np.argwhere(target_array == colour).min(0)))
    else:
        target_colours = list(range(10))
    source_specs: list[tuple[dict, int | None]] = [({}, None)]
    for colour in colours:
        semantic_specs = layer_role_candidates(g, colour, include_value=False)
        for spec in (semantic_specs or [{"value": int(colour)}]):
            source_specs.append((({"source_role": spec["role"]} if "role" in spec else {"source": spec["value"]}), colour))
    target_specs: list[tuple[dict, int]] = []
    input_colours = set(map(int, g.ravel()))
    for colour in target_colours:
        if colour in input_colours:
            semantic_specs = layer_role_candidates(g, colour, include_value=False)
            for spec in semantic_specs:
                target_specs.append(({"target_role": spec["role"]}, colour))
            if semantic_specs: continue
        target_specs.append(({"target": int(colour)}, int(colour)))
    steps, seen = [], set()

    def add(step):
        signature = tuple(sorted(step.items()))
        if signature not in seen:
            seen.add(signature); steps.append(step)

    same_shape = target_array is None or g.shape == target_array.shape
    input_foreground = int((g != bg).sum())
    target_foreground = input_foreground if target_array is None else int((target_array != target_bg).sum())
    for source_spec, source in source_specs:
        records = _component_records(g, source)
        effective_selectors, selected_signatures = [], set()
        for selector in OBJECT_SELECTORS:
            selected = _select_component_records(records, selector, g.shape)
            signature = tuple(sorted(cell for record in selected for cell in record["cells"]))
            if not signature or signature in selected_signatures: continue
            selected_signatures.add(signature); effective_selectors.append(selector)
        for selector in effective_selectors:
            base = {**source_spec, "selector": selector}
            for op in ("keep_selected", "delete_selected"):
                add({"op": op, **base})
            if not same_shape: add({"op": "crop_selected", **base})
            if same_shape:
                if target_foreground >= input_foreground:
                    add({"op": "fill_selected_bbox", **base}); add({"op": "outline_selected_bbox", **base})
                if target_foreground == input_foreground:
                    for transform in range(1, 8): add({"op": "transform_selected", **base, "transform": transform})
                    for direction in ("down", "up", "left", "right"):
                        add({"op": "gravity_selected", **base, "direction": direction})
            for target_spec, colour in target_specs:
                if source is None or colour != source:
                    add({"op": "recolor_selected", **base, **target_spec})
    return steps


def search_compositional_programs(task: dict, max_programs: int = 20,
                                  max_intermediates: int = 256, max_step_candidates: int = 512,
                                  time_limit_seconds: float | None = 2.0) -> list[dict]:
    """Bounded exact search over one- and two-stage object programs."""
    pairs = task["train"]; deadline = (time.perf_counter()+time_limit_seconds
                                       if time_limit_seconds is not None else float("inf"))
    if not pairs: return []
    inputs = [np.asarray(p["input"], dtype=np.int8) for p in pairs]
    targets = [np.asarray(p["output"], dtype=np.int8) for p in pairs]
    exact, exact_signatures = [], set()

    def add_exact(steps):
        signature = repr(steps)
        if signature not in exact_signatures:
            exact_signatures.add(signature); exact.append({"steps": steps})

    first_steps = enumerate_object_steps(inputs[0], targets[0])[:max_step_candidates]
    intermediates, intermediate_signatures = [], set()
    preparatory = {"keep_selected", "delete_selected", "recolor_selected", "crop_selected"}
    for step in first_steps:
        if time.perf_counter() >= deadline: break
        first_output = execute_object_step(inputs[0], step)
        if first_output is None: continue
        outputs = [first_output] + [execute_object_step(grid, step) for grid in inputs[1:]]
        if any(out is None for out in outputs): continue
        if np.array_equal(first_output, targets[0]) and all(
                np.array_equal(out, target) for out, target in zip(outputs[1:], targets[1:])):
            add_exact([step])
            if len(exact) >= max_programs: return exact
        if step["op"] not in preparatory: continue
        signature = tuple((out.shape, out.tobytes()) for out in outputs)
        if signature in intermediate_signatures: continue
        intermediate_signatures.add(signature); intermediates.append((step, outputs))
        if len(intermediates) >= max_intermediates: break

    for first, middle in intermediates:
        if time.perf_counter() >= deadline: break
        for second in enumerate_object_steps(middle[0], targets[0])[:max_step_candidates]:
            if time.perf_counter() >= deadline: break
            if second == first: continue
            first_output = execute_object_step(middle[0], second)
            if first_output is None or not np.array_equal(first_output, targets[0]): continue
            outputs = [execute_object_step(grid, second) for grid in middle[1:]]
            if any(out is None for out in outputs): continue
            if all(np.array_equal(out, target) for out, target in zip(outputs, targets[1:])):
                add_exact([first, second])
                if len(exact) >= max_programs: return exact
    return exact


@torch.no_grad()
def verified_relation_predictions(model: nn.Module, task: dict, device: torch.device,
                                  topk: int = 12) -> tuple[list[np.ndarray], list[dict]]:
    """Keep neural proposals only when their operation verifies on every demo.

    Source/context layer ids are re-predicted for each demonstration, allowing
    the same semantic role to use different categorical colours across pairs.
    """
    test_proposals = propose_relation_programs(model, task, device, topk)
    demo_proposals = []
    for pair in task["train"]:
        probe = {"train": task["train"], "test": [{"input": pair["input"]}]}
        demo_proposals.append(propose_relation_programs(model, probe, device, topk))
    outputs, traces, seen = [], [], set()
    for proposal in test_proposals:
        witnesses = []; valid = True
        for pair, proposals in zip(task["train"], demo_proposals):
            witness = None
            for candidate in proposals:
                if candidate["op"] != proposal["op"]: continue
                prediction = execute_relation_program(pair["input"], candidate)
                if prediction is not None and np.array_equal(prediction, np.asarray(pair["output"])):
                    witness = candidate; break
            if witness is None: valid = False; break
            witnesses.append(witness)
        if not valid: continue
        prediction = execute_relation_program(task["test"][0]["input"], proposal)
        if prediction is None: continue
        signature = (prediction.shape, prediction.tobytes())
        if signature in seen: continue
        seen.add(signature); outputs.append(prediction); traces.append({"test": proposal, "demo_witnesses": witnesses})
    return outputs, traces


In [ ]:
%%writefile arc_layer_operations.py
"""Categorical layer operations and bounded, demonstration-only ARC search.

No task IDs, stored answers, external APIs, or arbitrary Python execution.
The neural critic's feature schema and checkpoint are deliberately unchanged.
"""
from __future__ import annotations

from collections import Counter, deque
from itertools import combinations, zip_longest
import json
import time
from typing import Any

import numpy as np


OPERATION_FAMILIES = (
    "gravity_objects", "conditional_line_gravity", "connect_layer", "extend_rays",
    "complete_symmetry", "repeat_objects", "stamp_markers", "fill_enclosures", "panel_boolean",
    "follow_marker_path",
)
DIRECTIONS = {"down": 0, "left": 1, "up": 2, "right": 3}


def as_grid(value: Any) -> np.ndarray:
    grid = np.asarray(value)
    if (grid.ndim != 2 or not all(0 < n <= 30 for n in grid.shape)
            or not np.issubdtype(grid.dtype, np.number) or not np.isfinite(grid).all()
            or np.any(grid != np.floor(grid)) or np.any(grid < 0) or np.any(grid > 9)):
        raise ValueError("Expected a nonempty <=30x30 grid of categorical integers 0..9")
    return grid.astype(np.int8, copy=True)


def grid_key(grid: np.ndarray) -> tuple:
    return grid.shape, grid.tobytes()


def background(grid: np.ndarray) -> int:
    """Resolve ties spatially, never by the magnitude of a colour ID."""
    border = np.concatenate((grid[0], grid[-1], grid[:, 0], grid[:, -1]))
    edge_counts = Counter(map(int, border)); counts = Counter(map(int, grid.ravel()))
    first = {colour: int(np.flatnonzero(grid == colour)[0]) for colour in counts}
    return max(counts, key=lambda colour: (edge_counts[colour], counts[colour], -first[colour]))


def components(mask: np.ndarray, diagonal: bool = False) -> list[np.ndarray]:
    visited = np.zeros_like(mask, dtype=bool); groups = []
    h, w = mask.shape
    for row, col in np.argwhere(mask):
        if visited[row, col]:
            continue
        visited[row, col] = True; stack = [(int(row), int(col))]; cells = []
        while stack:
            r, c = stack.pop(); cells.append((r, c))
            neighbours = [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]
            if diagonal: neighbours += [(r-1, c-1), (r-1, c+1), (r+1, c-1), (r+1, c+1)]
            for nr, nc in neighbours:
                if 0 <= nr < h and 0 <= nc < w and mask[nr, nc] and not visited[nr, nc]:
                    visited[nr, nc] = True; stack.append((nr, nc))
        groups.append(np.asarray(cells, dtype=np.int16))
    return groups


def enclosed_mask(walls: np.ndarray) -> np.ndarray:
    free = ~walls; seen = np.zeros_like(walls, dtype=bool); h, w = walls.shape
    queue = deque()
    for r, c in np.argwhere(free):
        if r in (0, h-1) or c in (0, w-1):
            seen[r, c] = True; queue.append((int(r), int(c)))
    while queue:
        r, c = queue.popleft()
        for nr, nc in ((r-1, c), (r+1, c), (r, c-1), (r, c+1)):
            if 0 <= nr < h and 0 <= nc < w and free[nr, nc] and not seen[nr, nc]:
                seen[nr, nc] = True; queue.append((nr, nc))
    return free & ~seen


class GridContext:
    def __init__(self, grid: Any, background_colour: int | None = None):
        self.grid = as_grid(grid)
        self.bg = background(self.grid) if background_colour is None else int(background_colour)
        # First appearance is invariant under relabeling all ten colours.
        self.colours = list(dict.fromkeys(map(int, self.grid.ravel())))
        self.foreground = [c for c in self.colours if c != self.bg]
        self._components: dict[int, list[np.ndarray]] = {}
        self._roles: dict[str, int | None] | None = None

    def objects(self, colour: int) -> list[np.ndarray]:
        if colour not in self._components:
            self._components[colour] = components(self.grid == colour)
        return self._components[colour]

    def roles(self) -> dict[str, int | None]:
        if self._roles is not None:
            return self._roles
        areas = {c: int((self.grid == c).sum()) for c in self.foreground}
        counts = {c: len(self.objects(c)) for c in self.foreground}
        result: dict[str, int | None] = {"background": self.bg}
        result["only_foreground"] = self.foreground[0] if len(self.foreground) == 1 else None
        for name, values, choose in (("rarest", areas, min), ("most_frequent", areas, max),
                                      ("fewest_components", counts, min), ("most_components", counts, max)):
            chosen = [c for c, value in values.items() if value == choose(values.values())] if values else []
            result[name] = chosen[0] if len(chosen) == 1 else None
        if len(areas) >= 3:
            ranked = sorted(set(areas.values()), reverse=True)
            chosen = [c for c, value in areas.items() if len(ranked) > 1 and value == ranked[1]]
            result["second_largest_layer"] = chosen[0] if len(chosen) == 1 else None
        self._roles = result
        return result

    def resolve(self, specification: dict | None, *, default: int | None = None) -> int | None:
        if specification is None:
            return default
        if "role" in specification:
            return self.roles().get(specification["role"])
        value = specification.get("value")
        return int(value) if isinstance(value, (int, np.integer)) and 0 <= value <= 9 else None

    def specs(self, colour: int) -> list[dict]:
        semantic = [{"role": name} for name, value in self.roles().items() if value == colour]
        return semantic[:2] + [{"value": int(colour)}]


def _paint_background(out: np.ndarray, mask: np.ndarray, colour: int, bg: int) -> bool:
    if np.any(mask & (out != bg) & (out != colour)):
        return False
    out[mask] = colour
    return True


def _gravity_objects(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    if step.get("direction") not in DIRECTIONS:
        return None
    k = DIRECTIONS[step["direction"]]; grid = np.rot90(ctx.grid, k).copy(); bg = ctx.bg
    colours = ctx.foreground if source is None else [source]
    groups = [(colour, cells) for colour in colours for cells in components(grid == colour)]
    if not groups:
        return None
    # Front-most objects settle first; each island moves independently as a rigid body.
    groups.sort(key=lambda item: (-int(item[1][:, 0].max()), int(item[1][:, 1].min())))
    for colour, cells in groups:
        grid[cells[:, 0], cells[:, 1]] = bg
        delta = 0
        while True:
            shifted = cells + (delta+1, 0)
            if np.any(shifted[:, 0] >= grid.shape[0]) or np.any(grid[shifted[:, 0], shifted[:, 1]] != bg):
                break
            delta += 1
        shifted = cells + (delta, 0); grid[shifted[:, 0], shifted[:, 1]] = colour
    return np.rot90(grid, -k).copy()


def _conditional_line_gravity(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    marker = ctx.resolve(step.get("marker"))
    if source is None or marker is None or marker in (source, ctx.bg) or step.get("direction") not in DIRECTIONS:
        return None
    k = DIRECTIONS[step["direction"]]; grid = np.rot90(ctx.grid, k).copy(); out = grid.copy()
    active = (grid == source) | (grid == marker); h, w = grid.shape; lines = []
    for c in range(w):
        rows = np.flatnonzero(active[:, c]); runs = np.split(rows, np.flatnonzero(np.diff(rows) != 1)+1)
        for run in runs:
            if len(run) and np.any(grid[run, c] == source):
                values = grid[run, c].copy(); lines.append((c, run, values))
                out[run, c] = ctx.bg
    if not lines:
        return None
    landed = np.zeros_like(grid, dtype=bool)
    for c, rows, values in sorted(lines, key=lambda item: (-int(item[1][-1]), item[0])):
        delta = 0; stops_at_obstacles = bool(np.any(values == marker))
        while int(rows[-1])+delta+1 < h:
            shifted = rows+delta+1
            if np.any(landed[shifted, c]) or (stops_at_obstacles and np.any(out[shifted, c] != ctx.bg)):
                break
            delta += 1
        # Unmarked strips ignore static terrain, but erase only their final footprint.
        out[rows+delta, c] = values; landed[rows+delta, c] = True
    return np.rot90(out, -k).copy()


def _connect_layer(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    if source is None or step.get("axis") not in ("rows", "columns", "both"):
        return None
    out = ctx.grid.copy(); directions = (0, 1) if step["axis"] == "both" else ((0,) if step["axis"] == "rows" else (1,))
    for axis in directions:
        original = ctx.grid if axis == 0 else ctx.grid.T; view = out if axis == 0 else out.T
        for row in range(len(original)):
            positions = np.flatnonzero(original[row] == source)
            for left, right in zip(positions[:-1], positions[1:]):
                gap = view[row, left+1:right]
                if np.all((gap == ctx.bg) | (gap == source)):
                    gap[:] = source
    return out


def _extend_rays(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    if source is None or step.get("direction") not in DIRECTIONS:
        return None
    k = DIRECTIONS[step["direction"]]; grid = np.rot90(ctx.grid, k); out = grid.copy()
    for r, c in np.argwhere(grid == source):
        nr = int(r)+1
        while nr < len(grid) and grid[nr, c] == ctx.bg:
            out[nr, c] = source; nr += 1
    return np.rot90(out, -k).copy()


def _complete_symmetry(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    if source is None:
        return None
    mask = ctx.grid == source; coordinates = np.argwhere(mask)
    if not len(coordinates):
        return None
    if step.get("frame", "grid") == "bbox":
        r0, c0 = coordinates.min(0); r1, c1 = coordinates.max(0)+1
    else:
        r0 = c0 = 0; r1, c1 = mask.shape
    view = mask[r0:r1, c0:c1].copy(); mode = step.get("axis")
    if mode == "left_right": completed = view | np.fliplr(view)
    elif mode == "up_down": completed = view | np.flipud(view)
    elif mode == "rot180": completed = view | np.rot90(view, 2)
    elif mode == "both": completed = view | np.fliplr(view) | np.flipud(view) | np.rot90(view, 2)
    elif mode == "diagonal" and view.shape[0] == view.shape[1]: completed = view | view.T
    else: return None
    mask[r0:r1, c0:c1] = completed; out = ctx.grid.copy()
    return out if _paint_background(out, mask, source, ctx.bg) else None


def _repeat_objects(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    if source is None or step.get("axis") not in ("horizontal", "vertical"):
        return None
    groups = ctx.objects(source)
    if len(groups) < 2:
        return None
    anchors = [cells.min(0) for cells in groups]
    shapes = [frozenset(map(tuple, cells-anchor)) for cells, anchor in zip(groups, anchors)]
    if len(set(shapes)) != 1:
        return None
    axis = 1 if step["axis"] == "horizontal" else 0
    order = sorted(range(len(anchors)), key=lambda i: int(anchors[i][axis]))
    ordered = np.stack([anchors[i] for i in order]); deltas = np.diff(ordered, axis=0)
    if not np.all(deltas == deltas[0]) or deltas[0][axis] <= 0 or deltas[0][1-axis] != 0:
        return None
    delta = deltas[0]; cells = groups[order[0]]; out = ctx.grid.copy()
    indices = range(-30, 31) if step.get("scope") == "both" else range(0, 31)
    for n in indices:
        shifted = cells+n*delta
        if np.any(shifted < 0) or np.any(shifted[:, 0] >= out.shape[0]) or np.any(shifted[:, 1] >= out.shape[1]):
            continue
        existing = out[shifted[:, 0], shifted[:, 1]]
        if np.any((existing != ctx.bg) & (existing != source)):
            return None
        out[shifted[:, 0], shifted[:, 1]] = source
    return out


def _stamp_markers(ctx: GridContext, step: dict, source: int | None) -> np.ndarray | None:
    marker = ctx.resolve(step.get("marker"))
    if source is None or marker is None or marker in (source, ctx.bg):
        return None
    templates = ctx.objects(source); markers = ctx.objects(marker)
    if len(templates) != 1 or not markers or any(len(cells) != 1 for cells in markers):
        return None
    template = templates[0]; shape = template-template.min(0); out = ctx.grid.copy()
    anchor = step.get("anchor", "top_left")
    if anchor == "center":
        extent = shape.max(0)
        if np.any(extent % 2): return None
        shape = shape-extent//2
    elif anchor != "top_left": return None
    colour = marker if step.get("paint") == "marker" else source
    out[out == marker] = ctx.bg
    if not step.get("keep_template", True): out[out == source] = ctx.bg
    for point in markers:
        shifted = shape+point[0]
        if np.any(shifted < 0) or np.any(shifted[:, 0] >= out.shape[0]) or np.any(shifted[:, 1] >= out.shape[1]):
            return None
        values = out[shifted[:, 0], shifted[:, 1]]
        if np.any((values != ctx.bg) & (values != colour)):
            return None
        out[shifted[:, 0], shifted[:, 1]] = colour
    return out


def _panel_boolean(ctx: GridContext, step: dict) -> np.ndarray | None:
    axis, gap = step.get("axis"), step.get("gap", 1)
    if axis not in (0, 1) or gap not in (0, 1): return None
    size = ctx.grid.shape[axis]
    if size <= gap or (size-gap) % 2: return None
    half = (size-gap)//2
    if gap:
        separator = np.take(ctx.grid, half, axis=axis)
        if len(set(map(int, separator))) != 1: return None
    a, b = np.split(ctx.grid, (half, half+gap), axis=axis)[::2]
    left, right = a != ctx.bg, b != ctx.bg
    logic = step.get("logic")
    if logic == "or": mask = left | right
    elif logic == "and": mask = left & right
    elif logic == "xor": mask = left ^ right
    elif logic == "a_minus_b": mask = left & ~right
    elif logic == "b_minus_a": mask = right & ~left
    elif logic == "neither": mask = ~(left | right)
    else: return None
    colour = ctx.resolve(step.get("target")); bg = ctx.resolve(step.get("output_bg"), default=ctx.bg)
    if colour is None or bg is None: return None
    out = np.full(a.shape, bg, dtype=np.int8); out[mask] = colour
    return out


def _equal_panels(ctx: GridContext) -> list[np.ndarray] | None:
    choices = []
    for axis in (0, 1):
        lines = [i for i in range(ctx.grid.shape[axis])
                 if len(set(map(int, np.take(ctx.grid, i, axis=axis)))) == 1
                 and int(np.take(ctx.grid, i, axis=axis)[0]) != ctx.bg]
        if not lines: continue
        if len({int(np.take(ctx.grid, i, axis=axis)[0]) for i in lines}) != 1: continue
        bounds = [-1]+lines+[ctx.grid.shape[axis]]
        widths = [right-left-1 for left, right in zip(bounds[:-1], bounds[1:])]
        if min(widths) <= 0 or len(set(widths)) != 1: continue
        choices.append([np.take(ctx.grid, np.arange(left+1, right), axis=axis)
                        for left, right in zip(bounds[:-1], bounds[1:])])
    if len(choices) > 1: return None
    return choices[0] if choices else [ctx.grid]


def _trace_marker_path(ctx: GridContext, body: np.ndarray, markers: set[tuple]) -> np.ndarray | None:
    grid = ctx.grid; colour = ctx.foreground[0]; origin = body.min(0); normal = body-origin
    shape = tuple(normal.max(0)+1); mask = np.zeros(shape, bool); mask[normal[:, 0], normal[:, 1]] = True
    if len(components(mask, diagonal=True)) != 1: return None
    tips = []
    # An arrow has reflection symmetry around its heading and a single narrow tip.
    if np.array_equal(mask, np.flipud(mask)):
        left, right = int(mask[:, 0].sum()), int(mask[:, -1].sum())
        if left == 1 and left < right:
            tips.append(((0, -1), body[body[:, 1] == origin[1]][0]))
        if right == 1 and right < left:
            tips.append(((0, 1), body[body[:, 1] == body[:, 1].max()][0]))
    if np.array_equal(mask, np.fliplr(mask)):
        top, bottom = int(mask[0].sum()), int(mask[-1].sum())
        if top == 1 and top < bottom:
            tips.append(((-1, 0), body[body[:, 0] == origin[0]][0]))
        if bottom == 1 and bottom < top:
            tips.append(((1, 0), body[body[:, 0] == body[:, 0].max()][0]))
    if len(tips) != 1: return None
    heading, tip = tips[0]; offsets = body-tip; tip = tuple(map(int, tip)); remaining = set(markers)
    while remaining:
        by_direction = {}
        for point in remaining:
            delta = (point[0]-tip[0], point[1]-tip[1])
            if delta[0] == 0 and delta[1]: direction = (0, 1 if delta[1] > 0 else -1)
            elif delta[1] == 0 and delta[0]: direction = (1 if delta[0] > 0 else -1, 0)
            else: continue
            distance = abs(delta[0])+abs(delta[1])
            by_direction.setdefault(direction, []).append((distance, point))
        if heading in by_direction:
            next_heading = heading
        else:
            perpendicular = [direction for direction in by_direction
                             if direction[0]*heading[0]+direction[1]*heading[1] == 0]
            if len(perpendicular) != 1: return None
            next_heading = perpendicular[0]
        _, tip = min(by_direction[next_heading]); remaining.remove(tip)
        while heading != next_heading:
            offsets = np.column_stack((-offsets[:, 1], offsets[:, 0]))
            heading = (-heading[1], heading[0])
    final = offsets+np.asarray(tip)
    if np.any(final < 0) or np.any(final[:, 0] >= grid.shape[0]) or np.any(final[:, 1] >= grid.shape[1]): return None
    out = np.full_like(grid, ctx.bg); out[final[:, 0], final[:, 1]] = colour
    return out


def _follow_single_marker_path(grid: np.ndarray) -> np.ndarray | None:
    ctx = GridContext(grid)
    if len(ctx.foreground) != 1: return None
    colour = ctx.foreground[0]; foreground = set(map(tuple, np.argwhere(grid == colour)))
    if not 4 <= len(foreground) <= 60: return None
    groups = components(grid == colour, diagonal=True)
    bodies = [cells for cells in groups if 3 <= len(cells) <= 12]
    if not 1 <= len(bodies) <= 3: return None
    answers = []; largest_body = 0
    # A waypoint can touch an arrow diagonally. Separate at most two attached
    # points, then require a complete, unambiguous path consuming every marker.
    for connected in sorted(bodies, key=len, reverse=True):
        for removed in range(3):
            size = len(connected)-removed
            if size < max(3, largest_body): continue
            for omitted in combinations(range(len(connected)), removed):
                body = np.delete(connected, omitted, axis=0) if omitted else connected
                markers = foreground-set(map(tuple, body))
                if not markers: continue
                output = _trace_marker_path(ctx, body, markers)
                if output is not None:
                    if size > largest_body: answers = []; largest_body = size
                    answers.append(output)
    if not answers or any(not np.array_equal(answers[0], out) for out in answers[1:]): return None
    return answers[0]


def _follow_marker_path(ctx: GridContext) -> np.ndarray | None:
    panels = _equal_panels(ctx)
    if panels is None: return None
    outputs = [_follow_single_marker_path(panel) for panel in panels]
    if any(output is None for output in outputs): return None
    if any(not np.array_equal(outputs[0], output) for output in outputs[1:]): return None
    return outputs[0]


def execute_operation(grid: Any, step: dict, context: GridContext | None = None) -> np.ndarray | None:
    ctx = context if context is not None else GridContext(grid); op = step.get("op")
    if "input_bg" in step:
        explicit_bg = ctx.resolve(step["input_bg"])
        if explicit_bg is None: return None
        if explicit_bg != ctx.bg: ctx = GridContext(ctx.grid, background_colour=explicit_bg)
    source = ctx.resolve(step.get("source"))
    if "source" in step and (source is None or source == ctx.bg):
        return None
    if op == "gravity_objects": return _gravity_objects(ctx, step, source)
    if op == "conditional_line_gravity": return _conditional_line_gravity(ctx, step, source)
    if op == "connect_layer": return _connect_layer(ctx, step, source)
    if op == "extend_rays": return _extend_rays(ctx, step, source)
    if op == "complete_symmetry": return _complete_symmetry(ctx, step, source)
    if op == "repeat_objects": return _repeat_objects(ctx, step, source)
    if op == "stamp_markers": return _stamp_markers(ctx, step, source)
    if op == "panel_boolean": return _panel_boolean(ctx, step)
    if op == "follow_marker_path": return _follow_marker_path(ctx)
    if op == "fill_enclosures":
        target = ctx.resolve(step.get("target"))
        if target is None: return None
        walls = ctx.grid != ctx.bg if source is None else ctx.grid == source
        mask = enclosed_mask(walls) & (ctx.grid == ctx.bg); out = ctx.grid.copy(); out[mask] = target
        return out
    if op == "crop_foreground":
        points = np.argwhere(ctx.grid != ctx.bg)
        if not len(points): return None
        r0, c0 = points.min(0); r1, c1 = points.max(0)+1
        return ctx.grid[r0:r1, c0:c1].copy()
    return None


def execute_program(grid: Any, program: dict, context: GridContext | None = None) -> np.ndarray | None:
    steps = program.get("steps", [program])
    if not isinstance(steps, list) or not 1 <= len(steps) <= 2:
        return None
    current = grid
    for index, step in enumerate(steps):
        current = execute_operation(current, step, context if index == 0 else None)
        if current is None: return None
    return current


def program_key(program: dict) -> str:
    return json.dumps(program, sort_keys=True, separators=(",", ":"))


def program_cost(program: dict) -> int:
    steps = program.get("steps", [program])
    fixed = sum(isinstance(value, dict) and "value" in value for step in steps for value in step.values())
    return 100*len(steps) + 2*fixed + sum(len(step)-1 for step in steps)


def _enumerate_programs_raw(pairs: list[dict], *, composition: bool = True):
    """Infer routing/colour parameters from demonstrations, never test answers."""
    if not pairs: return
    ctx = GridContext(pairs[0]["input"]); target = as_grid(pairs[0]["output"])
    same_shape = ctx.grid.shape == target.shape
    families: dict[str, list[dict]] = {op: [] for op in OPERATION_FAMILIES}
    sources = [(spec, colour) for colour in ctx.foreground for spec in ctx.specs(colour)]
    target_bg = background(target)
    target_colours = [c for c in dict.fromkeys(map(int, target.ravel())) if c != target_bg]
    targets = [spec for colour in target_colours for spec in ctx.specs(colour)]
    families["follow_marker_path"].append({"op": "follow_marker_path"})
    if same_shape:
        for direction in DIRECTIONS:
            families["gravity_objects"].append({"op": "gravity_objects", "direction": direction})
        for source_spec, colour in sources:
            base = {"source": source_spec}
            for direction in DIRECTIONS:
                for op in ("gravity_objects", "extend_rays"):
                    families[op].append({"op": op, **base, "direction": direction})
            for axis in ("rows", "columns", "both"):
                families["connect_layer"].append({"op": "connect_layer", **base, "axis": axis})
            for axis in ("left_right", "up_down", "rot180", "both", "diagonal"):
                for frame in ("grid", "bbox"):
                    families["complete_symmetry"].append({"op": "complete_symmetry", **base, "axis": axis, "frame": frame})
            if len(ctx.objects(colour)) >= 2:
                for axis in ("horizontal", "vertical"):
                    for scope in ("forward", "both"):
                        families["repeat_objects"].append({"op": "repeat_objects", **base, "axis": axis, "scope": scope})
            for marker_spec, marker in sources:
                if marker == colour: continue
                for direction in DIRECTIONS:
                    families["conditional_line_gravity"].append({
                        "op": "conditional_line_gravity", **base, "marker": marker_spec, "direction": direction})
                if len(ctx.objects(colour)) == 1 and all(len(cells) == 1 for cells in ctx.objects(marker)):
                    for anchor in ("top_left", "center"):
                        for paint in ("source", "marker"):
                            for keep in (True, False):
                                families["stamp_markers"].append({"op": "stamp_markers", **base, "marker": marker_spec,
                                    "anchor": anchor, "paint": paint, "keep_template": keep})
        for source_spec in [None]+[spec for spec, _ in sources]:
            for target_spec in targets:
                step = {"op": "fill_enclosures", "target": target_spec}
                if source_spec is not None: step["source"] = source_spec
                families["fill_enclosures"].append(step)
    else:
        for axis in (0, 1):
            for gap in (0, 1):
                shape = list(ctx.grid.shape); shape[axis] = (shape[axis]-gap)//2
                if tuple(shape) != target.shape or (ctx.grid.shape[axis]-gap) % 2: continue
                for logic in ("or", "and", "xor", "a_minus_b", "b_minus_a", "neither"):
                    for target_spec in targets:
                        families["panel_boolean"].append({"op": "panel_boolean", "axis": axis, "gap": gap,
                            "logic": logic, "target": target_spec,
                            "output_bg": {"role": "background"} if target_bg == ctx.bg else {"value": target_bg}})
    # Round-robin families prevents candidate limits from starving later operations.
    for group in zip_longest(*families.values()):
        for step in group:
            if step is not None: yield {"steps": [step]}
    yield {"steps": [{"op": "crop_foreground"}]}
    if composition:
        cropped = [execute_operation(pair["input"], {"op": "crop_foreground"}) for pair in pairs]
        if all(grid is not None for grid in cropped) and any(
                grid.shape != np.shape(pair["input"]) for grid, pair in zip(cropped, pairs)):
            intermediate = [{"input": grid, "output": pair["output"]} for grid, pair in zip(cropped, pairs)]
            for program in _enumerate_programs_raw(intermediate, composition=False):
                if program["steps"][0]["op"] != "crop_foreground":
                    yield {"steps": [{"op": "crop_foreground"}]+program["steps"]}


def enumerate_programs(pairs: list[dict], *, composition: bool = True):
    if not pairs: return
    bg = background(as_grid(pairs[0]["input"]))
    for program in _enumerate_programs_raw(pairs, composition=composition):
        yield program
        # Compare spatially inferred and demonstration-constant backgrounds.
        # If they disagree on a query, the consensus gate must abstain.
        yield {"steps": [{**program["steps"][0], "input_bg": {"value": bg}}]+program["steps"][1:]}


def search_programs(task: dict, *, time_limit: float = 1.0, max_candidates: int = 4000) -> tuple[list[dict], dict]:
    start = time.perf_counter(); pairs = task.get("train", [])
    contexts = [GridContext(pair["input"]) for pair in pairs]
    targets = [as_grid(pair["output"]) for pair in pairs]
    exact, seen, tested, complete = [], set(), 0, True
    for program in enumerate_programs(pairs):
        if time.perf_counter()-start >= time_limit or tested >= max_candidates:
            complete = False; break
        signature = program_key(program)
        if signature in seen: continue
        seen.add(signature); tested += 1
        matches = True
        for ctx, target in zip(contexts, targets):
            output = execute_program(ctx.grid, program, ctx)
            if output is None or not np.array_equal(output, target):
                matches = False; break
        if matches:
            exact.append(program)
    exact.sort(key=lambda program: (program_cost(program), program_key(program)))
    return exact, {"tested": tested, "complete": complete, "elapsed": round(time.perf_counter()-start, 4), "exact": len(exact)}


def _minimal_consensus(programs: list[dict], query: Any) -> np.ndarray | None:
    if not programs: return None
    depth = len(programs[0]["steps"]); outputs = []; ctx = GridContext(query)
    for program in programs:
        # Do not let a tiny simplicity preference conceal a conflicting rule.
        if len(program["steps"]) != depth: continue
        output = execute_program(ctx.grid, program, ctx)
        if output is None: return None
        if outputs and not np.array_equal(output, outputs[0]): return None
        outputs.append(output)
    return outputs[0] if outputs else None


def verified_operation_predictions(task: dict, *, time_limit: float = 3.0) -> tuple[list[np.ndarray], dict]:
    """Fail closed on ambiguity/timeouts; perform independent leave-one-out search."""
    start = time.perf_counter(); pairs = task.get("train", []); queries = task.get("test", [])
    if len(pairs) < 2 or not queries: return [], {"accepted": False, "reason": "too_few_examples"}
    slice_budget = time_limit/(len(pairs)+1)
    programs, search_audit = search_programs(task, time_limit=slice_budget)
    audit = {"accepted": False, "full_search": search_audit, "loo": []}
    if not programs or not search_audit["complete"]:
        audit["reason"] = "no_exact_program" if not programs else "search_incomplete"
        return [], audit
    predictions = [_minimal_consensus(programs, pair["input"]) for pair in queries]
    if any(output is None for output in predictions):
        audit["reason"] = "query_ambiguity"; return [], audit
    for index, held_out in enumerate(pairs):
        remaining = time_limit-(time.perf_counter()-start)
        if remaining <= 0:
            audit["reason"] = "time_budget"; return [], audit
        fold = {"train": [pair for i, pair in enumerate(pairs) if i != index]}
        fold_programs, fold_audit = search_programs(fold, time_limit=min(slice_budget, remaining))
        prediction = _minimal_consensus(fold_programs, held_out["input"]) if fold_audit["complete"] else None
        passed = prediction is not None and np.array_equal(prediction, held_out["output"])
        audit["loo"].append({**fold_audit, "passed": bool(passed)})
        if not passed:
            audit["reason"] = "loo_failed"; return [], audit
    audit.update({"accepted": True, "reason": "exact_loo_consensus", "program": programs[0],
                  "elapsed": round(time.perf_counter()-start, 4)})
    return predictions, audit


In [ ]:
%%writefile operation_mining/panel_programs.py
"""Small, executable programs mined from categorical panel truth tables.

Used offline for discovery and by the experimental mining-transformer notebook.
Fitting consumes demonstration pairs only. Colours are labels; ordering applies
to panel indices, never to colour magnitudes. Keep every fitting hypothesis.
"""
from __future__ import annotations

from copy import deepcopy
from itertools import permutations, product
import json
import time

import numpy as np

from arc_layer_operations import as_grid


LOGICS = ("and", "or", "xor", "nor", "nand", "xnor")


def _colour(value):
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or not 0 <= value <= 9:
        raise ValueError("Expected a categorical colour label in 0..9")
    return int(value)


def split_panels(grid, partition):
    grid = as_grid(grid)
    axis, count = partition["axis"], partition["panel_count"]
    if axis not in (0, 1) or count not in (2, 3, 4):
        raise ValueError("Expected 2..4 panels and a spatial axis")
    if partition["split_mode"] == "equal":
        if grid.shape[axis] % count:
            raise ValueError("Panel sizes differ")
        panels = list(np.split(grid, count, axis=axis))
    elif partition["split_mode"] == "separator":
        colour = _colour(partition["separator_color"])
        separator = np.all(grid == colour, axis=1-axis).tolist()
        groups, start = [], None
        for index, is_separator in enumerate(separator + [True]):
            if not is_separator and start is None:
                start = index
            elif is_separator and start is not None:
                groups.append((start, index))
                start = None
        panels = [np.take(grid, range(a, b), axis=axis) for a, b in groups]
    else:
        raise ValueError("Unknown panel partition")
    if len(panels) != count or len({p.shape for p in panels}) != 1:
        raise ValueError("Partition must produce the requested equal-sized panels")
    return panels


def boolean_reduce(masks, logic):
    if logic == "and":
        return np.logical_and.reduce(masks)
    if logic == "or":
        return np.logical_or.reduce(masks)
    if logic == "xor":
        return np.logical_xor.reduce(masks)
    if logic == "nor":
        return ~np.logical_or.reduce(masks)
    if logic == "nand":
        return ~np.logical_and.reduce(masks)
    if logic == "xnor":
        return ~np.logical_xor.reduce(masks)
    raise ValueError("Unknown Boolean operation")


def _combine(panels, program):
    backgrounds = program["input_backgrounds"]
    if len(backgrounds) != len(panels):
        raise ValueError("One background binding is required per panel")
    masks = [p != _colour(b) for p, b in zip(panels, backgrounds)]
    output = np.full_like(panels[0], _colour(program["output_background"]))
    if program["op"] == "panel_boolean":
        output[boolean_reduce(masks, program["logic"])] = _colour(program["paint_colour"])
    elif program["op"] == "panel_priority_overlay":
        order = program["priority_low_to_high"]
        if sorted(order) != list(range(len(panels))):
            raise ValueError("Priority must be a permutation of panel indices")
        for index in order:
            output[masks[index]] = panels[index][masks[index]]
    else:
        raise ValueError("Unknown mined operation")
    return output


def execute(grid, program):
    return _combine(split_panels(grid, program["partition"]), program).astype(int).tolist()


def _table(rule):
    count = rule["panel_count"]
    if count not in (2, 3, 4):
        raise ValueError("Only 2..4 panels are supported")
    table = {}
    for entry in rule["mapping"]:
        values = tuple(_colour(v) for v in entry["values"])
        target = _colour(entry["output"])
        if len(values) != count or (values in table and table[values] != target):
            raise ValueError("Inconsistent panel truth table")
        table[values] = target
    if not table:
        raise ValueError("Empty panel truth table")
    return table


def compress_table(rule, *, deadline=None):
    """Synthesize Boolean/ordered-overlay hypotheses; do not choose by answers.

Certification is limited to the observed categorical input domain. An incomplete
table is explicitly marked; fitting it is not proof for unseen combinations.
The common background is a learned label binding, not a majority/zero heuristic.
"""
    if rule["kind"] != "panel_cell_map":
        raise ValueError("Expected a panel cell-map rule")
    table = _table(rule)
    count = rule["panel_count"]
    domains = [list(dict.fromkeys(v[i] for v in table)) for i in range(count)]
    backgrounds = [c for c in domains[0] if all(c in d for d in domains)]
    partition = {k: rule[k] for k in ("axis", "panel_count", "split_mode")}
    if partition["split_mode"] == "separator":
        partition["separator_color"] = _colour(rule["separator_color"])
    values = list(table)
    panels = [np.asarray([[v[i] for v in values]], dtype=np.int8) for i in range(count)]
    target = np.asarray([[table[v] for v in values]], dtype=np.int8)
    outputs = list(dict.fromkeys(table.values()))
    candidates = []

    def accept(program):
        if np.array_equal(_combine(panels, program), target):
            candidates.append({
                "program": program,
                "evidence": {
                    "observed_table_rows": len(table),
                    "domain_sizes": list(map(len, domains)),
                    "full_observed_domain": len(table) == int(np.prod(list(map(len, domains)))),
                    "scope": "categorical truth-table equivalence, not novel-task accuracy",
                },
            })

    for bg in backgrounds:
        if deadline is not None and time.monotonic() >= deadline:
            raise TimeoutError("Panel program search budget exhausted")
        common = {"partition": partition, "input_backgrounds": [bg]*count}
        if len(outputs) == 2:
            # Prefer a shared background binding for readable annotations, but
            # retain other equivalent encodings and all remaining hypotheses.
            output_backgrounds = [bg] + [c for c in outputs if c != bg] if bg in outputs else outputs
            for out_bg in output_backgrounds:
                paint = next(c for c in outputs if c != out_bg)
                for logic in LOGICS:
                    accept({**common, "op": "panel_boolean", "logic": logic,
                            "output_background": out_bg, "paint_colour": paint})
        for order in permutations(range(count)):
            if deadline is not None and time.monotonic() >= deadline:
                raise TimeoutError("Panel program search budget exhausted")
            accept({**common, "op": "panel_priority_overlay", "output_background": bg,
                    "priority_low_to_high": list(order)})
    return candidates


def _partition_candidates(grid):
    colours = list(dict.fromkeys(map(int, grid.ravel())))
    for count in (2, 3, 4):
        for axis in (0, 1):
            yield {"split_mode": "equal", "axis": axis, "panel_count": count}
            for colour in colours:
                yield {"split_mode": "separator", "axis": axis,
                       "panel_count": count, "separator_color": colour}


def fit_demonstrations(pairs, *, deadline=None):
    """Infer partition, categorical bindings, Boolean logic or layer priority.

No test/query input, query answer, task ID, or stored per-task rule is accepted.
All candidates fitting every demonstration are returned, including ambiguities.
"""
    if not pairs:
        return []
    arrays = [(as_grid(p["input"]), as_grid(p["output"])) for p in pairs]
    results, seen = [], set()
    for partition in _partition_candidates(arrays[0][0]):
        if deadline is not None and time.monotonic() >= deadline:
            raise TimeoutError("Panel program search budget exhausted")
        table = {}
        try:
            for inp, out in arrays:
                panels = split_panels(inp, partition)
                if panels[0].shape != out.shape:
                    raise ValueError("Output size differs")
                for values, colour in zip(zip(*(p.ravel() for p in panels)), out.ravel()):
                    values, colour = tuple(map(int, values)), int(colour)
                    if values in table and table[values] != colour:
                        raise ValueError("Output is not a spatially uniform panel operation")
                    table[values] = colour
            rule = {"kind": "panel_cell_map", **partition,
                    "mapping": [{"values": list(v), "output": c} for v, c in table.items()]}
            for candidate in compress_table(rule, deadline=deadline):
                program = candidate["program"]
                key = json.dumps(program, sort_keys=True)
                if key not in seen and all(execute(p["input"], program) == p["output"] for p in pairs):
                    results.append(candidate)
                    seen.add(key)
        except ValueError:
            continue
    return results


def consensus_prediction(grid, candidates):
    predictions = []
    for candidate in candidates:
        try:
            predictions.append(execute(grid, candidate["program"]))
        except ValueError:
            return None
    if predictions and all(p == predictions[0] for p in predictions[1:]):
        return predictions[0]
    return None


def recolour_program(program, permutation):
    """Conjugate categorical bindings for metamorphic colour-renaming tests."""
    if sorted(permutation) != list(range(10)):
        raise ValueError("Expected a permutation of all ten colour labels")
    result = deepcopy(program)
    for key in ("output_background", "paint_colour"):
        if key in result:
            result[key] = int(permutation[result[key]])
    result["input_backgrounds"] = [int(permutation[c]) for c in result["input_backgrounds"]]
    if "separator_color" in result["partition"]:
        result["partition"]["separator_color"] = int(permutation[result["partition"]["separator_color"]])
    return result


In [ ]:
%%writefile operation_mining/transformer_adapter.py
"""Trainable operation decoder on the existing, frozen layer/object transformer.

45 semantic annotations supervise a non-autoregressive sequence decoder. Only
the two mined panel program schemas have new executable parameter heads here;
the other annotations are auxiliary supervision, not 43 new executors.
The original 19-op model and its checkpoint format remain unchanged.
"""
from __future__ import annotations

from itertools import permutations
import math
import time

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from arc_layer_operations import as_grid
from layer_relation_transformer import (
    LayerRelationTransformer, extract_layer_graph, extract_object_graph,
)
from operation_mining.panel_programs import (
    LOGICS, consensus_prediction, execute, fit_demonstrations,
)


SCHEMA_VERSION = 1
PRIMITIVES = tuple(sorted((
    "split_panels", "categorical_layer_predicates", "compose_layers", "connected_components",
    "boolean_reduce", "transform_geometry", "crop_canvas", "feature_to_colour_lookup",
    "measure_geometry", "orient_copy_by_position", "paint_layers", "select_object", "tile_canvas",
    "paint_objects", "categorical_colour_lookup", "bounding_box", "paint_empty_cells",
    "panel_priority_overlay", "compose_objects", "detect_hollow_rectangle", "foreground_mask",
    "isolate_colour_layers", "measure_interior", "paint_enclosed_background", "resample_pixels",
    "background_mask", "draw_segment", "filter_solid_rectangles", "flood_from_border",
    "group_spatial_rows", "mask_difference", "move_until_collision", "objects_to_colour_matrix",
    "select_aligned_pairs", "sort_spatial_columns", "delete_object", "extend_ray", "filter_objects",
    "infer_constant_displacement", "paint_box", "repeat_until_border", "select_colour_role",
    "select_panel", "select_unique_panel_colour", "split_panel_grid",
)))
KINDS = ("annotation_only", "panel_boolean", "panel_priority_overlay")
PRIORITIES = tuple(permutations(range(4)))
MAX_STEPS = 8  # six-step source programs plus an explicit END, with one spare slot


def check_deadline(deadline):
    if deadline is not None and time.monotonic() >= deadline:
        raise TimeoutError("Mined operation budget exhausted")


class MinedOperationHead(nn.Module):
    def __init__(self, dim=128, depth=2, heads=4, max_steps=MAX_STEPS):
        super().__init__()
        self.max_steps = max_steps
        self.colour_fuse = nn.Sequential(nn.Linear(4*dim, dim), nn.GELU(), nn.Linear(dim, dim))
        self.layout_proj = nn.Sequential(nn.Linear(6, dim), nn.GELU(), nn.Linear(dim, dim))
        self.queries = nn.Parameter(torch.randn(1 + max_steps, dim) / math.sqrt(dim))
        layer = nn.TransformerDecoderLayer(dim, heads, 3*dim, dropout=.1, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, depth, norm=nn.LayerNorm(dim))
        self.sequence = nn.Linear(dim, len(PRIMITIVES) + 1)  # final token = END
        self.presence = nn.Linear(dim, len(PRIMITIVES))
        self.categorical = nn.ModuleDict({name: nn.Linear(dim, size) for name, size in (
            ("kind", 3), ("logic", len(LOGICS)), ("axis", 2), ("count", 3),
            ("partition", 2), ("priority", len(PRIORITIES)),
        )})
        self.colour_pointers = nn.ModuleDict({name: nn.Linear(dim, dim) for name in (
            "input_background", "output_background", "paint", "separator",
        )})

    def forward(self, batch):
        inp, out = batch["in_layers"], batch["out_layers"]
        colours = self.colour_fuse(torch.cat((inp, out, out-inp, inp*out), -1))
        memory = torch.cat((self.layout_proj(batch["layout"])[:, None], batch["demo_memory"], colours), 1)
        valid = torch.cat((torch.ones((len(inp), 1), dtype=torch.bool, device=inp.device),
                           batch["demo_mask"].bool(), batch["input_present"] | batch["output_present"]), 1)
        queries = self.queries[None].expand(len(inp), -1, -1)
        decoded = self.decoder(queries, memory, memory_key_padding_mask=~valid)
        context = decoded[:, 0]
        result = {name: head(context) for name, head in self.categorical.items()}
        result.update(sequence=self.sequence(decoded[:, 1:]), presence=self.presence(context))
        for name, query in self.colour_pointers.items():
            logits = torch.einsum("bd,bcd->bc", query(context), colours) / math.sqrt(colours.shape[-1])
            present = batch["output_present"] if name in {"output_background", "paint"} else batch["input_present"]
            result[name] = logits.masked_fill(~present, -1e4)
        return result


class MinedLayerTransformer(nn.Module):
    """The old encoder/critic plus a separately trained operation decoder."""
    def __init__(self, backbone, head_config=None):
        super().__init__()
        self.backbone = backbone
        self.head_config = head_config or {"dim": backbone.dim, "depth": 2, "heads": 4, "max_steps": MAX_STEPS}
        if self.head_config["dim"] != backbone.dim:
            raise ValueError("Decoder and backbone dimensions differ")
        self.mining_head = MinedOperationHead(**self.head_config)
        self.backbone.requires_grad_(False)
        self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()  # no dropout drift or forgetting in the existing model
        return self

    @property
    def device(self):
        return next(self.mining_head.parameters()).device

    @torch.no_grad()
    def encode_demonstrations(self, pairs, deadline=None):
        """Only demonstration I/O enters the neural model; no query answer or ID."""
        if not pairs:
            raise ValueError("At least one demonstration is required")
        self.backbone.eval()
        descriptors, layouts = [], []
        cache = {}
        for pair in pairs:
            sides = []
            for name in ("input", "output"):
                check_deadline(deadline)
                grid = as_grid(pair[name])
                key = (grid.shape, grid.tobytes())
                if key not in cache:
                    cache[key] = (*extract_layer_graph(grid), *extract_object_graph(grid))
                sides.append(cache[key])
            descriptors.append(sides)
            ih, iw = np.shape(pair["input"])
            oh, ow = np.shape(pair["output"])
            layouts.append([ih/30, iw/30, oh/30, ow/30, oh/ih, ow/iw])
        encoded, raw_nodes, raw_edges = [], [], []
        for side in (0, 1):
            values = [torch.as_tensor(np.stack([d[side][i] for d in descriptors]), device=self.device).unsqueeze(0)
                      for i in range(6)]
            encoded.append(self.backbone._encode_many(*values))
            raw_nodes.append(values[0]); raw_edges.append(values[1])
        (in_layers, ip), (out_layers, op) = encoded
        demo, _, _ = self.backbone._pair_embedding(in_layers, ip, out_layers, op,
                                                  raw_nodes[0], raw_edges[0], raw_nodes[1], raw_edges[1])
        in_present, out_present = raw_nodes[0][..., 0] > .5, raw_nodes[1][..., 0] > .5

        def pooled(layers, present):
            return (layers * present[..., None]).sum(1) / present.sum(1).clamp_min(1)[..., None]

        result = {
            "demo_memory": demo[0].detach().cpu(),
            "in_layers": pooled(in_layers, in_present)[0].detach().cpu(),
            "out_layers": pooled(out_layers, out_present)[0].detach().cpu(),
            "layout": torch.tensor(np.mean(layouts, axis=0), dtype=torch.float32),
            "input_present": in_present.any(1)[0].cpu(), "output_present": out_present.any(1)[0].cpu(),
        }
        check_deadline(deadline)
        return result

    def forward(self, memory_batch):
        return self.mining_head(memory_batch)

    @torch.no_grad()
    def infer(self, pairs, deadline=None):
        self.eval()
        features = self.encode_demonstrations(pairs, deadline)
        batch = collate_memories([features], self.device)
        outputs = self(batch)
        check_deadline(deadline)
        return {k: value.cpu() for k, value in outputs.items()}


def collate_memories(memories, device="cpu"):
    max_demo = max(len(m["demo_memory"]) for m in memories)
    dim = memories[0]["demo_memory"].shape[-1]
    batch = {"demo_memory": torch.zeros(len(memories), max_demo, dim),
             "demo_mask": torch.zeros(len(memories), max_demo, dtype=torch.bool)}
    for i, memory in enumerate(memories):
        size = len(memory["demo_memory"])
        batch["demo_memory"][i, :size] = memory["demo_memory"]
        batch["demo_mask"][i, :size] = True
    for name in ("in_layers", "out_layers", "layout", "input_present", "output_present"):
        batch[name] = torch.stack([m[name] for m in memories])
    return {k: v.to(device) for k, v in batch.items()}


def program_labels(program):
    partition = program["partition"]
    count = int(partition["panel_count"])
    if len(set(program["input_backgrounds"])) != 1:
        raise ValueError("This decoder currently supports a shared panel-background binding")
    labels = {"kind": KINDS.index(program["op"]), "axis": partition["axis"], "count": count-2,
              "partition": int(partition["split_mode"] == "separator"),
              "input_background": program["input_backgrounds"][0],
              "output_background": program["output_background"]}
    if labels["partition"]:
        labels["separator"] = partition["separator_color"]
    if program["op"] == "panel_boolean":
        labels.update(logic=LOGICS.index(program["logic"]), paint=program["paint_colour"])
    else:
        order = tuple(program["priority_low_to_high"]) + tuple(range(count, 4))
        labels["priority"] = PRIORITIES.index(order)
    return labels


def log_program_score(log_probs, program, index=0):
    return torch.stack([log_probs[name][index, label] for name, label in program_labels(program).items()]).sum()


def mining_loss(outputs, records):
    """Marginalize equivalent parameterizations instead of contradictory labels."""
    device = outputs["sequence"].device
    seq = torch.full(outputs["sequence"].shape[:2], -100, dtype=torch.long, device=device)
    presence = torch.zeros_like(outputs["presence"])
    parameter_losses = []
    logs = {k: v.log_softmax(-1) for k, v in outputs.items() if k not in {"sequence", "presence"}}
    for i, record in enumerate(records):
        tokens = [PRIMITIVES.index(p) for p in record["pipeline"]]
        if len(tokens) >= seq.shape[1]:
            raise ValueError("Pipeline exceeds decoder length; refusing silent truncation")
        seq[i, :len(tokens)+1] = torch.tensor(tokens + [len(PRIMITIVES)], device=device)
        presence[i, tokens] = 1
        alternatives = record["programs"]
        if alternatives:
            joint = torch.stack([log_program_score(logs, p, i) for p in alternatives])
            fields = min(len(program_labels(p)) for p in alternatives)
            parameter_losses.append(-torch.logsumexp(joint, 0) / fields)
        else:
            parameter_losses.append(-logs["kind"][i, 0])
    seq_loss = F.cross_entropy(outputs["sequence"].flatten(0, 1), seq.flatten())
    presence_loss = F.binary_cross_entropy_with_logits(outputs["presence"], presence)
    parameters = torch.stack(parameter_losses).mean()
    return seq_loss + .3*presence_loss + parameters, {"sequence": seq_loss, "presence": presence_loss, "parameters": parameters}


def decoded_pipeline(outputs, index=0):
    result = []
    for token in outputs["sequence"][index].argmax(-1).tolist():
        if token == len(PRIMITIVES):
            break
        result.append(PRIMITIVES[token])
    return result


def greedy_program(outputs, index=0):
    choice = {k: int(v[index].argmax()) for k, v in outputs.items() if k not in {"sequence", "presence"}}
    kind = KINDS[choice["kind"]]
    if kind == "annotation_only":
        return None
    count = choice["count"] + 2
    partition = {"axis": choice["axis"], "panel_count": count,
                 "split_mode": "separator" if choice["partition"] else "equal"}
    if choice["partition"]:
        partition["separator_color"] = choice["separator"]
    result = {"op": kind, "partition": partition, "input_backgrounds": [choice["input_background"]]*count,
              "output_background": choice["output_background"]}
    if kind == "panel_boolean":
        result.update(logic=LOGICS[choice["logic"]], paint_colour=choice["paint"])
    else:
        valid_ids = [i for i, p in enumerate(PRIORITIES) if p[count:] == tuple(range(count, 4))]
        best = max(valid_ids, key=lambda i: float(outputs["priority"][index, i]))
        result["priority_low_to_high"] = list(PRIORITIES[best][:count])
    return result


def propose_mined_programs(outputs, index=0, beam_size=32):
    """Typed neural parameter beam; no stored rules, grids, task IDs or answers."""
    if beam_size <= 0:
        return []
    logs = {k: v[index].log_softmax(-1) for k, v in outputs.items() if k not in {"sequence", "presence"}}
    beams = [(0.0, {})]
    # Binding-dependent validity avoids mixing incompatible marginal predictions.
    for field in ("kind", "axis", "count", "partition", "input_background", "output_background", "logic", "paint", "priority", "separator"):
        expanded = []
        for score, labels in beams:
            kind = labels.get("kind")
            if ((field in {"logic", "paint"} and kind != 1) or
                    (field == "priority" and kind != 2) or
                    (field == "separator" and labels.get("partition") != 1)):
                expanded.append((score, labels)); continue
            allowed = list(range(len(logs[field])))
            if field == "kind":
                allowed = [1, 2]
            elif field == "output_background" and kind == 2:
                allowed = [labels["input_background"]]
            elif field == "paint":
                allowed = [c for c in allowed if c != labels["output_background"]]
            elif field == "priority":
                count = labels["count"]+2
                allowed = [i for i, order in enumerate(PRIORITIES) if order[count:] == tuple(range(count, 4))]
            # Keep at most four values per field before global beam pruning.
            allowed.sort(key=lambda i: float(logs[field][i]), reverse=True)
            for value in allowed[:4]:
                if float(logs[field][value]) > -1000:
                    expanded.append((score+float(logs[field][value]), {**labels, field: value}))
        beams = sorted(expanded, key=lambda pair: pair[0], reverse=True)[:beam_size]
    result = []
    for score, labels in beams:
        count = labels["count"]+2
        partition = {"axis": labels["axis"], "panel_count": count,
                     "split_mode": "separator" if labels["partition"] else "equal"}
        if labels["partition"]:
            partition["separator_color"] = labels["separator"]
        program = {"op": KINDS[labels["kind"]], "partition": partition,
                   "input_backgrounds": [labels["input_background"]]*count,
                   "output_background": labels["output_background"]}
        if labels["kind"] == 1:
            program.update(logic=LOGICS[labels["logic"]], paint_colour=labels["paint"])
        else:
            program["priority_low_to_high"] = list(PRIORITIES[labels["priority"]][:count])
        result.append({"program": program, "neural_log_score": score})
    return result


def load_mined_transformer(path, device="cpu"):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if payload.get("mining_schema_version") != SCHEMA_VERSION or tuple(payload.get("mining_primitives", ())) != PRIMITIVES:
        raise ValueError("Missing or incompatible mining decoder; old checkpoints need adapter training")
    backbone = LayerRelationTransformer(**payload["config"])
    backbone.load_state_dict(payload["state_dict"], strict=True)
    model = MinedLayerTransformer(backbone, payload["mining_head_config"])
    model.mining_head.load_state_dict(payload["mining_head_state_dict"], strict=True)
    return model.to(device).eval(), payload


def verified_mining_predictions(model, task, time_limit=3.0):
    """Neural proposals/ranking + exhaustive bounded train-fit consensus and LOO.

Neural confidence NEVER resolves a disagreement between fitting programs. The
returned grids are from deterministic execution, not the annotation sequence.
"""
    started = time.monotonic()
    deadline = started + max(0.0, float(time_limit))
    audit = {"accepted": False, "neural_used": False}
    try:
        check_deadline(deadline)
        pairs = [{"input": p["input"], "output": p["output"]} for p in task["train"]]
        if len(pairs) < 2:
            return [], {**audit, "reason": "too_few_demonstrations"}
        candidates = fit_demonstrations(pairs, deadline=deadline)
        audit["fitting_programs"] = len(candidates)
        if not candidates:
            return [], {**audit, "reason": "no_panel_program"}
        for i, pair in enumerate(pairs):
            heldout_fit = fit_demonstrations([p for j, p in enumerate(pairs) if j != i], deadline=deadline)
            prediction = consensus_prediction(pair["input"], heldout_fit)
            if prediction is None or prediction != pair["output"]:
                return [], {**audit, "reason": "leave_one_demo_out_failed", "fold": i}
        predictions = [consensus_prediction(q["input"], candidates) for q in task["test"]]
        if not predictions or any(p is None for p in predictions):
            return [], {**audit, "reason": "query_ambiguity_or_invalid_partition"}
        if model is not None:
            outputs = model.infer(pairs, deadline)
            neural_proposals = propose_mined_programs(outputs)
            neural_fits = []
            for proposed in neural_proposals:
                check_deadline(deadline)
                try:
                    if all(execute(p["input"], proposed["program"]) == p["output"] for p in pairs):
                        neural_fits.append(proposed)
                except (ValueError, KeyError, IndexError):
                    continue
            # A verified neural proposal may have a binding not enumerated by
            # the table compressor. Include it in the ambiguity check, not just
            # the ranking. Never let confidence overrule a differing output.
            for proposed in neural_fits:
                if not any(c["program"] == proposed["program"] for c in candidates):
                    candidates.append(proposed)
            consensus = [consensus_prediction(q["input"], candidates) for q in task["test"]]
            if any(p is None for p in consensus):
                return [], {**audit, "reason": "neural_symbolic_disagreement", "neural_used": True}
            logs = {k: v.log_softmax(-1) for k, v in outputs.items() if k not in {"sequence", "presence"}}
            candidates.sort(key=lambda c: float(log_program_score(logs, c["program"])), reverse=True)
            proposal = greedy_program(outputs)
            try:
                proposal_fits = proposal is not None and all(execute(p["input"], proposal) == p["output"] for p in pairs)
            except (ValueError, KeyError, IndexError):
                proposal_fits = False
            audit.update(neural_used=True, neural_pipeline=decoded_pipeline(outputs),
                         neural_greedy_program=proposal, neural_greedy_fits_demonstrations=proposal_fits,
                         neural_beam_size=len(neural_proposals), neural_beam_demo_fits=len(neural_fits))
            # The decoder now chooses which verified program is executed, while
            # agreement across all fitting alternatives remains a hard gate.
            predictions = [execute(q["input"], candidates[0]["program"]) for q in task["test"]]
        check_deadline(deadline)
        audit.update(accepted=True, reason="exact_consensus_and_demo_holdouts",
                     ranked_program=candidates[0]["program"], seconds=round(time.monotonic()-started, 4))
        return [np.asarray(p, dtype=np.int8) for p in predictions], audit
    except TimeoutError:
        return [], {**audit, "reason": "mining_budget_exhausted", "seconds": round(time.monotonic()-started, 4)}


In [ ]:
%%writefile operation_mining/search_dsl.py
"""Executable grid-to-grid DSL for bounded neural-guided program search.

Colour IDs are categorical bindings, never magnitudes. Each intermediate is a
valid ARC grid. No task IDs, stored answers, eval solutions or Python eval.
"""
from __future__ import annotations

from collections import Counter
from itertools import permutations, product
import numpy as np

from arc_layer_operations import (OPERATION_FAMILIES, GridContext, as_grid,
    components, enclosed_mask, enumerate_programs, execute_operation)
from operation_mining.panel_programs import LOGICS, execute as execute_panel, split_panels

NEW_FAMILIES = (
    "rotate_grid", "reflect_grid", "translate_layer", "copy_layer",
    "recolor_layer", "swap_layers", "keep_objects", "delete_objects",
    "crop_objects", "recolor_objects", "fill_object_boxes", "outline_object_boxes",
    "fill_layer_holes", "scale_pixels", "reduce_blocks", "tile_grid", "mirror_tile",
    "trim_uniform_border", "select_panel", "panel_logic", "panel_overlay",
    "align_layers", "move_layer_until_touch",
)
FAMILIES = tuple(sorted((*OPERATION_FAMILIES, "crop_foreground", *NEW_FAMILIES)))
SELECTORS = ("all", "largest", "smallest", "border", "interior", "unique_shape",
             "repeated_shape", "topmost", "bottommost", "leftmost", "rightmost")
MAX_DEPTH = 3


def _selected(ctx, step):
    colour = ctx.resolve(step.get("source"))
    if "source" in step and (colour is None or colour == ctx.bg): return []
    records = [(c, p) for c in (ctx.foreground if colour is None else [colour]) for p in ctx.objects(c)]
    selector = step.get("selector", "all")
    if selector == "all": return records
    if not records: return []
    if selector in ("largest", "smallest"):
        size = (max if selector == "largest" else min)(len(p) for _, p in records)
        return [(c, p) for c, p in records if len(p) == size]
    if selector in ("border", "interior"):
        return [(c, p) for c, p in records if bool(np.any((p == 0) | (p == np.array(ctx.grid.shape)-1))) == (selector == "border")]
    if selector in ("unique_shape", "repeated_shape"):
        keys = [tuple(sorted(map(tuple, p-p.min(0)))) for _, p in records]
        counts = Counter(keys)
        return [record for record, key in zip(records, keys) if (counts[key] == 1) == (selector == "unique_shape")]
    if selector in ("topmost", "bottommost", "leftmost", "rightmost"):
        axis = 0 if selector in ("topmost", "bottommost") else 1
        extreme = max if selector in ("bottommost", "rightmost") else min
        values = [int(extreme(p[:, axis])) for _, p in records]
        return [record for record, value in zip(records, values) if value == extreme(values)]
    return []


def _shift(ctx, source, dr, dc, copy=False):
    cells = np.argwhere(ctx.grid == source)
    if not len(cells): return None
    dest = cells + (dr, dc)
    if np.any(dest < 0) or np.any(dest >= np.array(ctx.grid.shape)): return None
    out = ctx.grid.copy()
    if not copy: out[out == source] = ctx.bg
    if np.any((out[dest[:, 0], dest[:, 1]] != ctx.bg) & (out[dest[:, 0], dest[:, 1]] != source)): return None
    out[dest[:, 0], dest[:, 1]] = source
    return out


def execute_step(grid, step):
    """Return None for failed preconditions; never silently clip/overwrite layers."""
    try:
        g = as_grid(grid); op = step["op"]
        if op not in FAMILIES: return None
        if op in OPERATION_FAMILIES or op == "crop_foreground":
            result = execute_operation(g, step)
            return None if result is None else as_grid(result)
        ctx = GridContext(g); out = g.copy()
        source, target = ctx.resolve(step.get("source")), ctx.resolve(step.get("target"))
        if "source" in step and (source is None or source == ctx.bg): return None
        if "target" in step and target is None: return None
        if op == "rotate_grid": out = np.rot90(g, step["turns"])
        elif op == "reflect_grid":
            if step["axis"] == "transpose": out = g.T
            elif step["axis"] in ("rows", "columns"): out = np.flip(g, axis=0 if step["axis"] == "rows" else 1)
            else: return None
        elif op in ("translate_layer", "copy_layer"):
            out = _shift(ctx, source, int(step["dr"]), int(step["dc"]), op == "copy_layer")
        elif op in ("align_layers", "move_layer_until_touch"):
            if source is None or target is None or source == target: return None
            a, b = np.argwhere(g == source), np.argwhere(g == target)
            if not len(a) or not len(b): return None
            if op == "align_layers":
                axis = int(step["axis"])
                if axis not in (0, 1): return None
                anchor = step["anchor"]
                if anchor not in ("min", "max", "center"): return None
                aa = a.min(0) if anchor == "min" else a.max(0) if anchor == "max" else (a.min(0)+a.max(0))/2
                bb = b.min(0) if anchor == "min" else b.max(0) if anchor == "max" else (b.min(0)+b.max(0))/2
                delta = bb[axis]-aa[axis]
                if delta != int(delta): return None
                out = _shift(ctx, source, int(delta) if axis == 0 else 0, int(delta) if axis == 1 else 0)
            else:
                dr, dc = {"down": (1, 0), "up": (-1, 0), "left": (0, -1), "right": (0, 1)}[step["direction"]]
                out = None
                for distance in range(31):
                    candidate = _shift(ctx, source, dr*distance, dc*distance)
                    if candidate is None: break
                    moved = np.argwhere(candidate == source)
                    # Stop at four-neighbour contact, not overlap.
                    if min(np.abs(moved[:, None]-b[None]).sum(-1).ravel()) == 1:
                        out = candidate; break
        elif op == "recolor_layer":
            if source is None or target is None: return None
            out[g == source] = target
        elif op == "swap_layers":
            if source is None or target is None or not np.any(g == target): return None
            out[g == source] = target; out[g == target] = source
        elif op == "fill_layer_holes":
            if source is None: return None
            out[enclosed_mask(g == source) & (g == ctx.bg)] = source
        elif op in ("keep_objects", "delete_objects", "crop_objects", "recolor_objects", "fill_object_boxes", "outline_object_boxes"):
            records = _selected(ctx, step)
            if not records: return None
            cells = np.concatenate([p for _, p in records])
            if op in ("keep_objects", "crop_objects"):
                out[:] = ctx.bg; out[cells[:, 0], cells[:, 1]] = g[cells[:, 0], cells[:, 1]]
                if op == "crop_objects":
                    lo, hi = cells.min(0), cells.max(0)+1
                    out = out[lo[0]:hi[0], lo[1]:hi[1]]
            elif op == "delete_objects": out[cells[:, 0], cells[:, 1]] = ctx.bg
            elif op == "recolor_objects":
                if target is None: return None
                out[cells[:, 0], cells[:, 1]] = target
            else:
                for colour, points in records:
                    r0, c0 = points.min(0); r1, c1 = points.max(0)+1
                    mask = np.zeros_like(g, dtype=bool)
                    mask[r0:r1, c0:c1] = True
                    if op == "outline_object_boxes": mask[r0+1:r1-1, c0+1:c1-1] = False
                    if np.any(mask & (out != ctx.bg) & (out != colour)): return None
                    out[points[:, 0], points[:, 1]] = ctx.bg; out[mask] = colour
        elif op in ("scale_pixels", "reduce_blocks", "tile_grid", "mirror_tile"):
            rh, rw = int(step["rows"]), int(step["columns"])
            if not 1 <= rh <= 4 or not 1 <= rw <= 4: return None
            if op == "reduce_blocks":
                h, w = g.shape
                if h % rh or w % rw: return None
                out = g[::rh, ::rw]
                if not np.array_equal(np.repeat(np.repeat(out, rh, 0), rw, 1), g): return None
            else:
                if g.shape[0]*rh > 30 or g.shape[1]*rw > 30: return None
                if op == "scale_pixels": out = np.repeat(np.repeat(g, rh, 0), rw, 1)
                elif op == "tile_grid": out = np.tile(g, (rh, rw))
                else:
                    out = np.concatenate([np.concatenate([np.flip(g, axis=tuple(a for a, n in ((0, r), (1, c)) if n % 2)) for c in range(rw)], 1) for r in range(rh)], 0)
        elif op == "trim_uniform_border":
            colour = g[0, 0]
            while min(out.shape) > 2 and all(np.all(edge == colour) for edge in (out[0], out[-1], out[:, 0], out[:, -1])):
                out = out[1:-1, 1:-1]
        elif op in ("select_panel", "panel_logic", "panel_overlay"):
            if op == "select_panel": out = split_panels(g, step["partition"])[int(step["index"])]
            else:
                native = {**step, "op": "panel_boolean" if op == "panel_logic" else "panel_priority_overlay"}
                out = execute_panel(g, native)
        else: return None
        return None if out is None else as_grid(out)
    except (ValueError, TypeError, KeyError, IndexError):
        return None


def execute_program(grid, program):
    steps = program.get("steps", [])
    if not 1 <= len(steps) <= MAX_DEPTH: return None
    current = grid
    for step in steps:
        current = execute_step(current, step)
        if current is None: return None
    return current


def candidate_groups(pairs, per_family=192):
    """Finite parameter proposals, NOT a successful-program fit prerequisite.

    Intermediate steps need not have the final output shape or pixel count.
    All parameter bindings originate from remaining demo I/O only.
    """
    ctx = GridContext(pairs[0]["input"]); g = ctx.grid; target_grid = as_grid(pairs[0]["output"])
    groups = {name: [] for name in FAMILIES}; seen = set()
    def add(step):
        from json import dumps
        key = dumps(step, sort_keys=True)
        family = step["op"]
        if key not in seen and len(groups[family]) < per_family:
            seen.add(key); groups[family].append(step)
    # Inherited families also run against shape-neutral targets for compositions.
    for view in (pairs, [{"input": p["input"], "output": p["input"]} for p in pairs]):
        for program in enumerate_programs(view, composition=False): add(program["steps"][0])
    for turns in (1, 2, 3): add({"op": "rotate_grid", "turns": turns})
    for axis in ("rows", "columns", "transpose"): add({"op": "reflect_grid", "axis": axis})
    add({"op": "trim_uniform_border"})
    colours = list(dict.fromkeys(map(int, target_grid.ravel())))
    source_specs = [s for c in ctx.foreground for s in ctx.specs(c)]
    target_specs = [s for c in colours for s in ctx.specs(c)]
    for source in source_specs:
        for op in ("fill_layer_holes",): add({"op": op, "source": source})
        for target in target_specs:
            for op in ("recolor_layer", "swap_layers"): add({"op": op, "source": source, "target": target})
        for target in source_specs:
            if source == target: continue
            for axis, anchor in product((0, 1), ("min", "max", "center")):
                add({"op": "align_layers", "source": source, "target": target, "axis": axis, "anchor": anchor})
            for direction in ("down", "up", "left", "right"):
                add({"op": "move_layer_until_touch", "source": source, "target": target, "direction": direction})
        # Small displacements plus an observed centroid displacement when integral.
        deltas = [(dr, dc) for dr, dc in product(range(-2, 3), repeat=2) if dr or dc]
        c = ctx.resolve(source)
        a, b = np.argwhere(g == c), np.argwhere(target_grid == c)
        if len(a) and len(b):
            d = b.mean(0)-a.mean(0)
            if np.allclose(d, d.astype(int)): deltas.insert(0, tuple(map(int, d)))
        for dr, dc in deltas:
            for op in ("translate_layer", "copy_layer"): add({"op": op, "source": source, "dr": dr, "dc": dc})
    for source in [None]+source_specs:
        for selector in SELECTORS:
            base = {"selector": selector, **({"source": source} if source else {})}
            for op in ("keep_objects", "delete_objects", "crop_objects", "fill_object_boxes", "outline_object_boxes"):
                add({"op": op, **base})
            for target in target_specs: add({"op": "recolor_objects", **base, "target": target})
    factors = [(h, w) for h, w in product(range(1, 5), repeat=2) if (h, w) != (1, 1)]
    # Shape ratios are geometric parameters, not colour arithmetic.
    factors.sort(key=lambda hw: (tuple(np.array(g.shape)*hw) != target_grid.shape, hw[0] != hw[1], sum(hw), hw))
    for rows, columns in factors:
        for op in ("scale_pixels", "reduce_blocks", "tile_grid", "mirror_tile"):
            add({"op": op, "rows": rows, "columns": columns})
    partitions = []
    for axis, count in product((0, 1), (2, 3, 4)):
        for part in [{"split_mode": "equal", "axis": axis, "panel_count": count}]+[
            {"split_mode": "separator", "axis": axis, "panel_count": count, "separator_color": c} for c in ctx.colours]:
            try: panels = split_panels(g, part)
            except ValueError: continue
            partitions.append((panels[0].shape != target_grid.shape, part, panels))
    for _, part, panels in sorted(partitions, key=lambda p: p[0]):
        count = len(panels)
        for index in range(count): add({"op": "select_panel", "partition": part, "index": index})
        common = [c for c in ctx.colours if all(np.any(p == c) for p in panels)]
        if ctx.bg in common: common.remove(ctx.bg); common.insert(0, ctx.bg)
        for bg in common:
            for output_bg in list(dict.fromkeys([GridContext(target_grid).bg]+colours)):
                for paint, logic in product(colours, LOGICS):
                    if paint != output_bg:
                        add({"op": "panel_logic", "partition": part, "input_backgrounds": [bg]*count,
                             "output_background": output_bg, "paint_colour": paint, "logic": logic})
            for priority in permutations(range(count)):
                add({"op": "panel_overlay", "partition": part, "input_backgrounds": [bg]*count,
                     "output_background": bg, "priority_low_to_high": list(priority)})
    return groups


def scheduled_steps(groups, probabilities=None):
    """Weighted fair enumeration: exploration never entirely disappears."""
    order = list(FAMILIES) if probabilities is None else sorted(FAMILIES, key=lambda n: -probabilities[FAMILIES.index(n)])
    queues = {name: iter(groups[name]) for name in order}; active = set(order)
    while active:
        for rank, name in enumerate(order):
            if name not in active: continue
            quota = (8 if rank == 0 else 5 if rank == 1 else 3 if rank == 2 else 1) if probabilities is not None else 1
            for _ in range(quota):
                try: yield next(queues[name])
                except StopIteration: active.remove(name); break


In [ ]:
%%writefile operation_mining/search_policy.py
"""Next-operation transformer conditioned on current demo states and history."""
from __future__ import annotations
import time
import numpy as np
import torch
from torch import nn

from layer_relation_transformer import LayerRelationTransformer, extract_layer_graph, extract_object_graph
from arc_layer_operations import as_grid
from operation_mining.transformer_adapter import collate_memories
from operation_mining.search_dsl import FAMILIES, MAX_DEPTH

SEARCH_SCHEMA = 1


class NextOperationHead(nn.Module):
    def __init__(self, dim=128, depth=2, heads=4):
        super().__init__()
        self.colour_fuse = nn.Sequential(nn.Linear(dim*4, dim), nn.GELU(), nn.Linear(dim, dim))
        self.layout = nn.Linear(6, dim)
        self.history = nn.Embedding(len(FAMILIES)+1, dim)
        self.position = nn.Parameter(torch.randn(MAX_DEPTH, dim)*.02)
        self.query = nn.Parameter(torch.randn(1, 1, dim)*.02)
        layer = nn.TransformerDecoderLayer(dim, heads, 3*dim, dropout=.1, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, depth, norm=nn.LayerNorm(dim))
        self.operation = nn.Linear(dim, len(FAMILIES))

    def forward(self, batch, history):
        inp, out = batch["in_layers"], batch["out_layers"]
        colours = self.colour_fuse(torch.cat((inp, out, out-inp, inp*out), -1))
        history_tokens = self.history(history)+self.position[None, :history.shape[1]]
        memory = torch.cat((self.layout(batch["layout"])[:, None], batch["demo_memory"], colours, history_tokens), 1)
        valid = torch.cat((torch.ones((len(inp), 1), dtype=torch.bool, device=inp.device), batch["demo_mask"],
                           batch["input_present"] | batch["output_present"], history != len(FAMILIES)), 1)
        context = self.decoder(self.query.expand(len(inp), -1, -1), memory, memory_key_padding_mask=~valid)[:, 0]
        return self.operation(context)


class SearchPolicy(nn.Module):
    def __init__(self, backbone, config=None):
        super().__init__()
        self.backbone = backbone
        self.config = config or {"dim": backbone.dim, "depth": 2, "heads": 4}
        self.policy = NextOperationHead(**self.config)
        self.backbone.requires_grad_(False).eval()
        self._grid_cache = {}

    @property
    def device(self): return next(self.policy.parameters()).device

    def train(self, mode=True):
        super().train(mode); self.backbone.eval(); return self

    def clear_cache(self): self._grid_cache.clear()

    @torch.no_grad()
    def encode_demonstrations(self, pairs, deadline=None):
        """Cache individual frozen grid embeddings, NEVER task rules/answers.

        Reused output grids are common across partial states and LOO folds.
        No pooled full-task context survives into a fold. Only the grids in the
        supplied pairs are assembled into that fold's memory.
        """
        if not pairs: raise ValueError('At least one demonstration is required')
        self.backbone.eval(); descriptors = []; layouts = []
        def check():
            if deadline is not None and time.monotonic() >= deadline: raise TimeoutError
        for pair in pairs:
            sides = []
            for name in ('input', 'output'):
                check(); grid = as_grid(pair[name]); key = (str(self.device), grid.shape, grid.tobytes())
                if key not in self._grid_cache:
                    values = [torch.as_tensor(v, device=self.device)[None, None] for v in (*extract_layer_graph(grid), *extract_object_graph(grid))]
                    layers, pool = self.backbone._encode_many(*values)
                    if len(self._grid_cache) >= 256: self._grid_cache.clear()
                    self._grid_cache[key] = (layers, pool, values[0], values[1])
                sides.append(self._grid_cache[key])
            descriptors.append(sides)
            ih, iw = np.shape(pair['input']); oh, ow = np.shape(pair['output'])
            layouts.append([ih/30, iw/30, oh/30, ow/30, oh/ih, ow/iw])
        encoded = [[torch.cat([p[side][field] for p in descriptors], 1) for field in range(4)] for side in (0, 1)]
        il, ip, inn, ine = encoded[0]; ol, op, outn, oute = encoded[1]
        demo, _, _ = self.backbone._pair_embedding(il, ip, ol, op, inn, ine, outn, oute)
        inp, outp = inn[..., 0] > .5, outn[..., 0] > .5
        def pool(layers, present): return ((layers*present[..., None]).sum(1)/present.sum(1).clamp_min(1)[..., None])[0].cpu()
        check()
        return {'demo_memory': demo[0].cpu(), 'in_layers': pool(il, inp), 'out_layers': pool(ol, outp),
                'layout': torch.tensor(np.mean(layouts, 0), dtype=torch.float32),
                'input_present': inp.any(1)[0].cpu(), 'output_present': outp.any(1)[0].cpu()}

    def forward(self, memories, histories): return self.policy(memories, histories)

    @torch.no_grad()
    def probabilities(self, pairs, history, deadline=None):
        self.eval()
        features = self.encode_demonstrations(pairs, deadline)
        histories = history_tensor([history], self.device)
        logits = self(collate_memories([features], self.device), histories)
        if deadline is not None and time.monotonic() >= deadline: raise TimeoutError
        # An explicit exploration floor; this model ranks families, NOT parameters.
        return (.95*logits.softmax(-1)[0]+.05/len(FAMILIES)).cpu().numpy()


def history_tensor(histories, device="cpu"):
    result = torch.full((len(histories), MAX_DEPTH), len(FAMILIES), dtype=torch.long, device=device)
    for index, history in enumerate(histories):
        if len(history) > MAX_DEPTH: raise ValueError("History exceeds DSL depth")
        for pos, item in enumerate(history):
            result[index, pos] = FAMILIES.index(item["op"] if isinstance(item, dict) else item)
    return result


def load_search_policy(path, device="cpu"):
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if payload.get("search_schema") != SEARCH_SCHEMA or tuple(payload.get("search_families", ())) != FAMILIES:
        raise ValueError("Checkpoint is not trained for this executable search DSL")
    backbone = LayerRelationTransformer(**payload["config"])
    backbone.load_state_dict(payload["state_dict"], strict=True)
    model = SearchPolicy(backbone, payload["search_config"])
    model.policy.load_state_dict(payload["search_state_dict"], strict=True)
    return model.to(device).eval(), payload


In [ ]:
%%writefile operation_mining/guided_search.py
"""Transformer-first bounded beam search; demonstration validation is mandatory.

Search is approximate, not an exhaustive proof of uniqueness. Independent LOO
search and query consensus are additional safeguards, not generalization proof.
"""
from __future__ import annotations
from dataclasses import dataclass
import json
import time
import numpy as np

from arc_layer_operations import as_grid, grid_key
from operation_mining.search_dsl import FAMILIES, MAX_DEPTH, candidate_groups, execute_step, execute_program, scheduled_steps


@dataclass(frozen=True)
class SearchConfig:
    max_depth: int = 3
    beam_width: int = 3
    per_state: int = 128
    max_expansions: int = 640
    per_family: int = 192

    def __post_init__(self):
        if not 1 <= self.max_depth <= MAX_DEPTH or min(self.beam_width, self.per_state, self.max_expansions, self.per_family) < 1:
            raise ValueError("Invalid bounded search configuration")


def _quality(grids, targets):
    values = []
    for g, t in zip(grids, targets):
        # Guidance only: NEVER require monotonic pixel improvement.
        shape = 1-np.abs(np.array(g.shape)-t.shape).sum()/60
        hist = 1-np.abs(np.bincount(g.ravel(), minlength=10)/g.size-np.bincount(t.ravel(), minlength=10)/t.size).sum()/2
        pixel = float(np.mean(g == t)) if g.shape == t.shape else 0
        values.append(.2*shape+.2*hist+.6*pixel)
    return float(np.mean(values))


def search(pairs, model=None, *, config=None, deadline=None):
    config = config or SearchConfig(); started = time.monotonic()
    audit = {"neural_forward_calls": 0, "expanded_states": 0, "tested_steps": 0,
             "valid_transitions": 0, "exact_programs": 0, "timed_out": False,
             "first_families": [], "mode": "neural" if model is not None else "uniform",
             "exhaustive": False}
    if not pairs: return [], audit
    states = [(tuple(as_grid(p["input"]) for p in pairs), [], 0.0)]
    targets = tuple(as_grid(p["output"]) for p in pairs)
    exact = []; seen_programs = set()
    def check():
        if deadline is not None and time.monotonic() >= deadline: raise TimeoutError
    try:
        for depth in range(config.max_depth):
            successors = []; seen_states = {}
            for current, prefix, log_prior in states:
                check()
                if audit["tested_steps"] >= config.max_expansions: break
                residual = [{"input": g, "output": t} for g, t in zip(current, targets)]
                # Crucial ordering: no symbolic fit/gate before this forward pass.
                probabilities = None
                if model is not None:
                    audit["neural_forward_calls"] += 1
                    probabilities = np.asarray(model.probabilities(residual, prefix, deadline), dtype=float)
                    if probabilities.shape != (len(FAMILIES),) or not np.isfinite(probabilities).all() or np.any(probabilities < 0):
                        raise ValueError("Invalid neural family probabilities")
                    if probabilities.sum() <= 0: raise ValueError("Empty neural distribution")
                    probabilities /= probabilities.sum()
                check(); groups = candidate_groups(residual, config.per_family); check()
                audit["expanded_states"] += 1
                if not prefix:
                    audit["first_families"] = sorted(FAMILIES, key=lambda name: -probabilities[FAMILIES.index(name)])[:5] if probabilities is not None else list(FAMILIES[:5])
                for index, step in enumerate(scheduled_steps(groups, probabilities)):
                    check()
                    if index >= config.per_state or audit["tested_steps"] >= config.max_expansions: break
                    audit["tested_steps"] += 1
                    outputs = tuple(execute_step(g, step) for g in current)
                    check()
                    if any(g is None for g in outputs): continue
                    if all(np.array_equal(a, b) for a, b in zip(outputs, current)): continue
                    audit["valid_transitions"] += 1
                    steps = prefix+[step]; program = {"steps": steps}
                    if all(np.array_equal(g, t) for g, t in zip(outputs, targets)):
                        key = json.dumps(program, sort_keys=True)
                        if key not in seen_programs:
                            seen_programs.add(key); exact.append(program)
                        continue
                    if depth+1 == config.max_depth: continue
                    signature = tuple(grid_key(g) for g in outputs)
                    # Keep two equivalent demo states: some programs differ on queries.
                    if seen_states.get(signature, 0) >= 2: continue
                    seen_states[signature] = seen_states.get(signature, 0)+1
                    prior = log_prior+(np.log(max(probabilities[FAMILIES.index(step["op"])], 1e-9)) if probabilities is not None else -np.log(len(FAMILIES)))
                    score = _quality(outputs, targets)+.04*prior/len(steps)
                    successors.append((score, outputs, steps, prior))
            # Shortest-depth exact programs are retained together for consensus.
            if exact or audit["tested_steps"] >= config.max_expansions: break
            successors.sort(key=lambda row: -row[0])
            # Diversity by first operation, then fill the remaining beam positions.
            selected, roots = [], set()
            for row in successors:
                root = row[2][0]["op"]
                if root not in roots:
                    selected.append(row); roots.add(root)
                if len(selected) == config.beam_width: break
            for row in successors:
                if len(selected) >= config.beam_width: break
                if not any(row is old for old in selected): selected.append(row)
            states = [(grids, steps, prior) for _, grids, steps, prior in selected]
            if not states: break
    except TimeoutError:
        audit["timed_out"] = True
        exact = []  # fail closed on deadline exhaustion, including a late forward
    audit.update(exact_programs=len(exact), elapsed=round(time.monotonic()-started, 5),
                 expansion_cap_reached=audit["tested_steps"] >= config.max_expansions)
    return exact, audit


def consensus(programs, grid):
    if not programs: return None
    outputs = [execute_program(grid, program) for program in programs]
    if any(g is None for g in outputs) or any(not np.array_equal(g, outputs[0]) for g in outputs[1:]): return None
    return outputs[0]


def verified_search_predictions(model, task, *, time_limit=4.0, config=None):
    started = time.monotonic(); deadline = started+max(0, time_limit)
    # Strip IDs/metadata/test outputs before passing ANY data to search/model.
    pairs = [{"input": p["input"], "output": p["output"]} for p in task.get("train", [])]
    queries = [{"input": p["input"]} for p in task.get("test", [])]
    audit = {"accepted": False, "reason": "too_few_examples", "loo": [], "neural_forward_calls": 0}
    if len(pairs) < 2 or not queries: return [], audit
    if model is not None and hasattr(model, 'clear_cache'): model.clear_cache()
    # Reserve verification time, but do not waste three quarters of the budget
    # on folds that will never run if no program can be found in the first place.
    programs, first = search(pairs, model, config=config, deadline=min(deadline, time.monotonic()+time_limit*.6))
    audit["search"] = first; audit["neural_forward_calls"] += first["neural_forward_calls"]
    if not programs:
        audit["reason"] = "search_timeout" if first["timed_out"] else "no_exact_program"
        return [], audit
    predictions = [consensus(programs, p["input"]) for p in queries]
    if any(g is None for g in predictions): audit["reason"] = "query_ambiguity"; return [], audit
    for i, held in enumerate(pairs):
        fold = [p for j, p in enumerate(pairs) if j != i]
        slice_seconds = max(0, deadline-time.monotonic())/(len(pairs)-i)
        fitted, detail = search(fold, model, config=config, deadline=min(deadline, time.monotonic()+slice_seconds))
        guess = consensus(fitted, held["input"])
        passed = guess is not None and np.array_equal(guess, held["output"])
        audit["loo"].append({**detail, "passed": bool(passed)})
        audit["neural_forward_calls"] += detail["neural_forward_calls"]
        if not passed: audit["reason"] = "loo_failed"; return [], audit
    if time.monotonic() >= deadline: audit["reason"] = "verification_timeout"; return [], audit
    audit.update(accepted=True, reason="bounded_exact_loo_consensus", program=programs[0],
                 elapsed=round(time.monotonic()-started, 5))
    return predictions, audit


In [ ]:

from pathlib import Path

def find_search_checkpoint():
    for name in ("layer_relation_transformer_search_v3.pt", "layer_relation_transformer.pt"):
        for root in (Path('/kaggle/working'), Path.cwd()):
            if (root/name).is_file(): return root/name
        if Path('/kaggle/input').exists():
            matches = sorted(Path('/kaggle/input').rglob(name))
            if matches: return matches[0]
    return None

_relation_checkpoint_path = find_search_checkpoint()
print('search checkpoint:', _relation_checkpoint_path)


In [ ]:

import math
import os
import numpy as np
import torch

from arc_decoder import score_kgmon
from layer_relation_transformer import LayerRelationTransformer, score_relation_candidates


RELATION_MODEL = None
RELATION_DEVICE = torch.device("cpu")
RELATION_TOP_K = int(os.getenv("ARC_RELATION_TOP_K", "8"))
RELATION_MIN_SCORE = float(os.getenv("ARC_RELATION_MIN_SCORE", "0.62"))
RELATION_MIN_MARGIN = float(os.getenv("ARC_RELATION_MIN_MARGIN", "0.10"))
RELATION_CALIBRATION_MARGIN = float(os.getenv("ARC_RELATION_CALIBRATION_MARGIN", "0.08"))

try:
    _payload = torch.load(_relation_checkpoint_path, map_location="cpu", weights_only=False)
    RELATION_MODEL = LayerRelationTransformer(**_payload.get("config", {"dim": 128, "depth": 3, "heads": 4}))
    RELATION_MODEL.load_state_dict(_payload["state_dict"], strict=True)
    RELATION_MODEL.to(RELATION_DEVICE).eval()
    print("layer relation critic loaded on", RELATION_DEVICE)
except Exception as exc:
    RELATION_MODEL = None
    print("layer relation critic disabled; original Qwen selection is preserved:", repr(exc))


def _valid_negative_grids(pair):
    """Create simple wrong outputs only for critic calibration, never as predictions."""
    inp = np.asarray(pair["input"], dtype=np.int8)
    out = np.asarray(pair["output"], dtype=np.int8)
    negatives = []
    for candidate in (inp, np.flipud(out), np.fliplr(out), np.rot90(out)):
        candidate = np.asarray(candidate, dtype=np.int8)
        if candidate.ndim == 2 and 0 < candidate.shape[0] <= 30 and 0 < candidate.shape[1] <= 30:
            if not np.array_equal(candidate, out) and not any(np.array_equal(candidate, old) for old in negatives):
                negatives.append(candidate)
        if len(negatives) == 2:
            break
    return negatives


@torch.no_grad()
def calibrate_relation_critic(task):
    """Check whether the critic recognizes held-out known outputs for this task."""
    if RELATION_MODEL is None or len(task.get("train", [])) < 2:
        return False, {"reason": "model_missing_or_too_few_demos"}
    margins, positive_scores = [], []
    train_pairs = task["train"]
    for held_out, pair in enumerate(train_pairs):
        negatives = _valid_negative_grids(pair)
        if not negatives:
            continue
        calibration_task = {
            "train": [p for i, p in enumerate(train_pairs) if i != held_out],
            "test": [{"input": pair["input"]}],
        }
        candidates = [np.asarray(pair["output"], dtype=np.int8)] + negatives
        try:
            scores = score_relation_candidates(
                RELATION_MODEL, calibration_task, candidates, RELATION_DEVICE)
        except Exception:
            continue
        positive_scores.append(scores[0])
        margins.append(scores[0] - max(scores[1:]))

    required = max(1, math.ceil(0.60 * len(train_pairs)))
    if len(margins) < required:
        return False, {"reason": "insufficient_calibration_pairs", "used": len(margins), "required": required}
    median_margin = float(np.median(margins))
    median_positive = float(np.median(positive_scores))
    pass_count = sum(m >= RELATION_CALIBRATION_MARGIN for m in margins)
    reliable = (
        pass_count >= required
        and median_margin >= RELATION_CALIBRATION_MARGIN
        and median_positive >= RELATION_MIN_SCORE
    )
    return reliable, {
        "used": len(margins),
        "passed": pass_count,
        "median_margin": round(median_margin, 4),
        "median_positive": round(median_positive, 4),
    }


@torch.no_grad()
def relation_safe_order(task, ordered_outputs):
    """Preserve attempt_1; replace attempt_2 only after strict task-local calibration."""
    ordered = list(ordered_outputs)
    if RELATION_MODEL is None or len(ordered) < 3:
        return ordered, {"changed": False, "reason": "critic_unavailable_or_too_few_candidates"}

    reliable, calibration = calibrate_relation_critic(task)
    if not reliable:
        return ordered, {"changed": False, "reason": "calibration_failed", "calibration": calibration}

    candidates = ordered[:RELATION_TOP_K]
    try:
        scores = np.asarray(
            score_relation_candidates(RELATION_MODEL, task, candidates, RELATION_DEVICE),
            dtype=np.float32,
        )
    except Exception as exc:
        return ordered, {"changed": False, "reason": "candidate_scoring_failed", "error": repr(exc)}

    best_index = int(np.argmax(scores))
    if best_index <= 1:
        return ordered, {"changed": False, "reason": "original_top2_supported", "calibration": calibration}

    best_score = float(scores[best_index])
    second_score = float(scores[1])
    first_score = float(scores[0])
    required_reference = max(first_score, second_score)
    if best_score < RELATION_MIN_SCORE or best_score - required_reference < RELATION_MIN_MARGIN:
        return ordered, {
            "changed": False,
            "reason": "critic_margin_too_small",
            "scores": [round(first_score, 4), round(second_score, 4), round(best_score, 4)],
            "calibration": calibration,
        }

    # attempt_1 stays byte-for-byte the original Qwen winner.
    reranked = [ordered[0], ordered[best_index]]
    reranked.extend(value for i, value in enumerate(ordered) if i not in (0, best_index))
    return reranked, {
        "changed": True,
        "promoted_from_rank": best_index + 1,
        "scores": [round(first_score, 4), round(second_score, 4), round(best_score, 4)],
        "calibration": calibration,
    }


def run_hybrid_selection(decoder):
    selected, audit = {}, {}
    for basekey, basevalues in decoder.decoded_results.items():
        original_order = score_kgmon(basevalues)
        try:
            task = decoder.dataset.queries[basekey]
            selected[basekey], audit[basekey] = relation_safe_order(task, original_order)
        except Exception as exc:
            selected[basekey] = original_order
            audit[basekey] = {"changed": False, "reason": "safe_fallback", "error": repr(exc)}
    changed = sum(bool(item.get("changed")) for item in audit.values())
    calibrated = sum(item.get("reason") != "calibration_failed" for item in audit.values())
    print(f"relation critic: changed attempt_2 for {changed}/{len(audit)} queries; audit entries={len(audit)}")
    return selected, audit


In [ ]:

import time
from arc_layer_operations import OPERATION_FAMILIES, verified_operation_predictions

OPERATION_SECONDS_PER_TASK = float(os.getenv("ARC_OPERATION_SECONDS_PER_TASK", "3"))
OPERATION_TOTAL_SECONDS = float(os.getenv("ARC_OPERATION_TOTAL_SECONDS", "240"))


def _same_grid(left, right):
    return bool(np.array_equal(left, right))


def promote_verified_operation(original, current, proposal):
    """Keep the actual Qwen winner, not merely the current critic winner."""
    if not original:
        return [proposal, proposal], True
    if _same_grid(proposal, original[0]):
        return list(current), False
    if len(current) > 1 and _same_grid(proposal, current[1]):
        return list(current), False
    ordered = [original[0], proposal]
    for candidate in current:
        if not any(_same_grid(candidate, previous) for previous in ordered):
            ordered.append(candidate)
    return ordered, True


In [ ]:

from operation_mining.search_policy import SearchPolicy, SEARCH_SCHEMA
from operation_mining.search_dsl import FAMILIES
from operation_mining.guided_search import SearchConfig, verified_search_predictions

SEARCH_MODEL = None
SEARCH_SECONDS = float(os.getenv('ARC_SEARCH_SECONDS', '4'))
SEARCH_TOTAL_SECONDS = float(os.getenv('ARC_SEARCH_TOTAL_SECONDS', '240'))
SEARCH_CONFIG = SearchConfig(max_depth=3, beam_width=3, per_state=128, max_expansions=640)
try:
    if RELATION_MODEL is None: raise ValueError('Missing relation backbone')
    if _payload.get('search_schema') != SEARCH_SCHEMA or tuple(_payload.get('search_families', ())) != FAMILIES:
        raise ValueError('Attach layer_relation_transformer_search_v3.pt; previous checkpoints have no next-operation policy')
    SEARCH_MODEL = SearchPolicy(RELATION_MODEL, _payload['search_config'])
    SEARCH_MODEL.policy.load_state_dict(_payload['search_state_dict'], strict=True)
    SEARCH_MODEL.to(RELATION_DEVICE).eval()
    torch.set_num_threads(min(torch.get_num_threads(), 2))
    print('Transformer-first search:', len(FAMILIES), 'executable families, depth <= 3')
except Exception as exc:
    SEARCH_MODEL = None
    print('Search policy disabled; previous Qwen/critic/v2 fallback remains:', repr(exc))


def run_search_hybrid(decoder, original_dataset):
    baseline = decoder.run_selection_algo()
    critic_only, v2_only, combined, audit, pools = {}, {}, {}, {}, {}
    with_arcgen = {}
    cache = {}; deadline = time.monotonic()+SEARCH_TOTAL_SECONDS
    # The Qwen runner reserves 600 seconds. Keep a final serialization margin.
    if 'global_end_time' in globals():
        deadline = min(deadline, time.monotonic()+max(0, global_end_time+540-time.time()))
    for basekey, query_task in decoder.dataset.queries.items():
        original = list(baseline.get(basekey, []))
        try: current, critic_audit = relation_safe_order(query_task, original)
        except Exception as exc: current, critic_audit = original, {'changed': False, 'reason': 'critic_error', 'error': repr(exc)}
        critic_only[basekey] = list(current)
        task_id, query_index = basekey.rsplit('_', 1); query_index = int(query_index)
        if task_id not in cache:
            proposed, prior = [], []
            search_audit = {'accepted': False, 'reason': 'model_missing', 'neural_forward_calls': 0}
            prior_audit = {'accepted': False, 'reason': 'global_operation_budget'}
            try:
                raw = original_dataset.queries[task_id]
                clean = {'train': [{'input': p['input'], 'output': p['output']} for p in raw['train']],
                         'test': [{'input': p['input']} for p in raw['test']]}
                # New policy gets the FIRST search call, independently of symbolic success.
                remaining = max(0, deadline-time.monotonic())
                if SEARCH_MODEL is not None and remaining > 0:
                    try:
                        proposed, search_audit = verified_search_predictions(SEARCH_MODEL, clean,
                            time_limit=min(SEARCH_SECONDS, remaining), config=SEARCH_CONFIG)
                    except Exception as exc:
                        search_audit = {'accepted': False, 'reason': 'search_error', 'error': repr(exc)}
                elif SEARCH_MODEL is not None:
                    search_audit['reason'] = 'global_operation_budget'
                # Existing v2 acts as a separate conservative fallback/check.
                remaining = max(0, deadline-time.monotonic())
                if remaining > 0:
                    try: prior, prior_audit = verified_operation_predictions(clean, time_limit=min(OPERATION_SECONDS_PER_TASK, remaining))
                    except Exception as exc: prior_audit = {'accepted': False, 'reason': 'v2_error', 'error': repr(exc)}
            except Exception as exc:
                search_audit = {'accepted': False, 'reason': 'task_input_error', 'error': repr(exc)}
            cache[task_id] = proposed, search_audit, prior, prior_audit
        proposed, search_audit, prior, prior_audit = cache[task_id]
        prior_changed = search_changed = False
        if prior_audit.get('accepted') and query_index < len(prior):
            current, prior_changed = promote_verified_operation(original, current, prior[query_index])
        v2_only[basekey] = list(current)
        # Preserve an accepted v2 prediction if the new branch disagrees.
        if not prior_audit.get('accepted') and search_audit.get('accepted') and query_index < len(proposed):
            current, search_changed = promote_verified_operation(original, current, proposed[query_index])
        no_arcgen = list(current)
        arcgen_changed, arcgen_audit = False, {'accepted': False, 'reason': 'disabled_or_missing'}
        _ag = globals().get('ARCGEN_RESULTS', {}).get(task_id)
        if _ag is not None:
            arcgen_audit = dict(_ag['audit'])
            if (_ag['accepted'] and not prior_audit.get('accepted') and not search_audit.get('accepted')
                    and query_index < len(_ag['predictions'])):
                current, arcgen_changed = promote_verified_operation(original, current, _ag['predictions'][query_index])
        if original: assert current and _same_grid(current[0], original[0]), 'Qwen attempt_1 must stay unchanged'
        combined[basekey] = no_arcgen; with_arcgen[basekey] = current
        audit[basekey] = {'critic': critic_audit, 'search': search_audit, 'v2': prior_audit,
                         'search_changed_attempt_2': search_changed, 'v2_changed_attempt_2': prior_changed,
                         'arcgen': arcgen_audit, 'arcgen_changed_attempt_2': arcgen_changed,
                         'attempt_1_preserved': not original or _same_grid(current[0], original[0])}
        pools[basekey] = {'qwen_top8': [np.asarray(p).tolist() for p in original[:8]],
                          'v2_top2': [np.asarray(p).tolist() for p in v2_only[basekey][:2]],
                          'hybrid_top2': [np.asarray(p).tolist() for p in current[:2]]}
    nonempty = sum(bool(v) for v in baseline.values())
    coverage = {'tasks': len(cache), 'queries': len(audit), 'qwen_nonempty_queries': nonempty,
                'neural_forward_calls': sum(row[1].get('neural_forward_calls', 0) for row in cache.values()),
                'search_accepted_tasks': sum(bool(row[1].get('accepted')) for row in cache.values()),
                'arcgen_tasks': len(globals().get('ARCGEN_RESULTS', {})),
                'arcgen_accepted_tasks': sum(bool(r.get('accepted')) for r in globals().get('ARCGEN_RESULTS', {}).values()),
                'full_public_eval_requested': os.getenv('ARC_FULL_PUBLIC_EVAL', '1') == '1',
                'note': 'Scores are raw task-credit sums, not percentages. Incomplete Qwen coverage is not a full comparison.'}
    print('Coverage and actual neural activity:', coverage)
    return baseline, critic_only, v2_only, combined, audit, pools, coverage, with_arcgen


In [ ]:
from arcgen.kaggle_branch import load_results

ARCGEN_RESULTS = {}
if ARCGEN_ENABLED:
    _limit = time.time() + float(os.getenv("ARC_ARCGEN_WAIT_SECONDS", "600"))
    if "global_end_time" in globals():
        _limit = min(_limit, global_end_time + 400)
    while ARCGEN_PROC is not None and ARCGEN_PROC.poll() is None and time.time() < _limit:
        time.sleep(5)
    if ARCGEN_PROC is not None and ARCGEN_PROC.poll() is None:
        print("arcgen branch still running; using the partial results written so far")
        ARCGEN_PROC.terminate()
    ARCGEN_RESULTS = load_results(ARCGEN_OUT)
    print(f"arcgen branch: {len(ARCGEN_RESULTS)} tasks processed, "
          f"{sum(r['accepted'] for r in ARCGEN_RESULTS.values())} accepted")

In [ ]:

import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv('KAGGLE_IS_COMPETITION_RERUN')
competition_root = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
if rerun_mode:
    data = ArcDataset.from_file(str(competition_root/'arc-agi_test_challenges.json'))
else:
    data = ArcDataset.from_file(str(competition_root/'arc-agi_evaluation_challenges.json'))
    data = data.load_replies(str(competition_root/'arc-agi_evaluation_solutions.json'))
decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results('/kaggle/inference_outputs')
baseline, critic, previous, hybrid, audit, pools, coverage, hybrid_arcgen = run_search_hybrid(decoder, data)
submissions = {'baseline': data.get_submission(baseline), 'critic_only': data.get_submission(critic),
               'layer_operations_v2': data.get_submission(previous), 'transformer_search_v3': data.get_submission(hybrid),
               'transformer_search_v3_arcgen': data.get_submission(hybrid_arcgen)}
for filename, value in (
    ('submission.json', submissions['transformer_search_v3_arcgen' if ARCGEN_ENABLED else 'transformer_search_v3']),
    ('submission_v3_without_arcgen.json', submissions['transformer_search_v3']),
    ('submission_baseline.json', submissions['baseline']),
    ('submission_critic_only.json', submissions['critic_only']),
    ('submission_layer_operations_v2.json', submissions['layer_operations_v2']),
    ('layer_search_audit.json', audit), ('qwen_candidate_pool.json', pools), ('run_coverage.json', coverage),
):
    with open(filename, 'w') as file: json.dump(value, file)
if not rerun_mode:
    scores = {name: data.validate_submission(value) for name, value in submissions.items()}
    print('Same-run raw task-credit sums (not percentages):', scores)
    with open('paired_selection_scores.json', 'w') as file: json.dump(scores, file, indent=2)
